# LIVE Laya inference test — v6 BRUTAL (Laya 421M + v3 graph, 230-question bank)

**What this is:** a LIVE run of the Laya classification model through the
v3 family-law decision graph, on (a) the same 16 layman questions used in the
local backtest, now scored against the CORRECTED v2 ground truth (D1-A: HMA 27
removed from K0/H3/H10/H15, GT 57 → 53 sections), (b) the FROZEN brutal bank
of 230 questions in 10 batches (B1–B9 of 25, B10 of 5), and (c) the sealed
fresh-20 and Kannada-8 sets. Stage 1 is one Laya typed decision per issue,
recorded with the model's actual chosen option and probabilities. Stage 2 is
the graph recheck (divorce gate first, HMA 24 dropped with no proceeding
pending, HMA 25 dropped with no decree, custody to exactly one primary route,
Stridhan co-fire civil-return + BNS 316, residence PWDVA 17+18+19, quantum
always escalated, never invented, D2: HMA 27 dropped with no proceeding
pending or being filed). Scoring: exact section-set match, per-section
precision / recall, forbidden mappings, court correct, escalates correctly —
reported BOTH as the old boolean AND as typed-kinds equality.

**Graph v3:** 34 sections / 48 nodes / 79 edges (Lane A: +hma-9, +pwdva-22).
Expected SHA-256: 67bf2b04ca946f7c7fa3062068e01ce051fbce8f4d740c281afe6fff841c34a5.

**NO TUNING between batches.** The notebook, builder, graph, thresholds,
prompts, phrasebook and fact annotations are frozen for the whole brutal run.
A failure found in batch 2 is NOT fixed before batch 3. All 10 batches run,
the report is filed, and only then is a fix diagnosed — in general terms.
No brutal question, phrase, or paraphrase may enter the phrasebook, prompts,
examples, or threshold fittings.

**Prose is NOT MEASURED in v6** (no Cell 9 renderer — see build note):
Fp is NOT MEASURED, so any rating quoted from this run is capped at 9.0.
The fresh-20 score is a MEASURED NUMBER ONLY, not a gate pass, until the
G2 20/20 amendment is signed (see the sealed cells and the build note).

**How to run:** upload this notebook to Google Drive (account used before:
nivedhbhat15@gmail.com), open it in Colab, upload
`laya-crpc-bns-section-decision-tree-v3.json` when the graph cell asks,
upload `brutal-bank-v1-frozen.json` when the bank cell asks, then
Runtime > Run all. Each batch cell downloads its JSON immediately — file it
before starting the next batch. A crash loses at most one batch.


In [ ]:
# Cell 1 — environment check (no install yet).
import os, platform, sys
print('python:', sys.version.split()[0], '| platform:', platform.machine())
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f'RAM total: {vm.total/1e9:.1f} GB, available: {vm.available/1e9:.1f} GB')
except Exception as e:
    print('psutil not available:', e)
    with open('/proc/meminfo') as f:
        print([l for l in f if 'MemTotal' in l or 'MemAvailable' in l])
try:
    import torch
    print('torch:', torch.__version__, '| cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0),
              f'| VRAM: {torch.cuda.get_device_properties(0).total/1e9:.1f} GB')
except Exception as e:
    print('torch not importable yet (install cell will add it):', e)
print()
print('Colab free tier is ~12-13 GB RAM; the 421M model needs ~1.7 GB FP32 / ~0.84 GB FP16,')
print('so it fits easily on CPU RAM and on a T4 GPU. Runtime > Change runtime type > T4 GPU is')
print('optional but faster; CPU-only also works.')


In [ ]:
# Cell 2 — install dependencies.
# *** APPROVAL REQUIRED: running this cell installs and runs upstream code
# (the `laya` package and its dependencies) inside this Colab session.
# Run it only if you approve installing/running upstream laya code here
# in Colab. This does NOT install anything on the local VM.
# laya is pinned to 0.5.5 because that is the version whose metadata was
# inspected locally on 2026-11-26 (the version the local blockers were
# found against). Do not unpin it without re-checking the load API.
!pip install -q "laya==0.5.5" torch transformers safetensors huggingface_hub
import importlib
for pkg in ['torch', 'transformers', 'safetensors', 'huggingface_hub', 'laya']:
    try:
        m = importlib.import_module(pkg)
        print(pkg, '->', getattr(m, '__version__', 'installed (no __version__)'))
    except Exception as e:
        print(pkg, 'NOT importable:', e)


In [ ]:
# Cell 3 — inspect the laya API BEFORE loading any model.
# The snapshot's test.py uses the old entry point laya.load('.'), but
# laya 0.5.5 loads a new Vega engine by default via laya.Agent(), and
# classic checkpoints need explicit naming. This cell finds out which
# entry points actually exist in the installed version, so the load
# cell below does not silently test the wrong model.
import inspect
import laya
print('laya version:', getattr(laya, '__version__', 'unknown'))
print('has laya.load:', hasattr(laya, 'load'))
print('has laya.Agent:', hasattr(laya, 'Agent'))
print('has laya.Router:', hasattr(laya, 'Router'))
for name in ['load', 'Agent', 'Router']:
    obj = getattr(laya, name, None)
    if obj is not None:
        try:
            print(f'--- {name} signature ---')
            print(inspect.signature(obj))
        except Exception as e:
            print(f'{name}: signature not available:', e)
        print(f'{name} doc (first 500 chars):', (obj.__doc__ or '')[:500])


In [ ]:
# Cell 4 — download the model snapshot and load the CLASSIC encoder
# checkpoint ONLY. Never load the Vega engine (the 0.8B reader that
# laya.Agent() loads by default) — that would test a different model.
# Order of attempts: (1) laya.load(local snapshot dir) if it exists,
# (2) laya.Agent(local dir, vegaml=False), (3) laya.Agent(
# 'convaiinnovations/laya', vegaml=False). The first loader that returns
# a classic checkpoint (architectures LayaTypedDecisions / ModernBERT
# encoder) wins; every failure is printed, none is hidden.
import os, time, json
os.environ['HF_HUB_OFFLINE'] = '0'  # download ON for this cell only
from huggingface_hub import snapshot_download

t0 = time.perf_counter()
model_dir = None
try:
    model_dir = snapshot_download(repo_id='PIXMELT/laya')
    print('downloaded PIXMELT/laya ->', model_dir)
except Exception as e:
    print('PIXMELT/laya snapshot_download FAILED:', repr(e))
    print('Will try the named upstream checkpoint via laya.Agent instead.')
    model_dir = None

if model_dir:
    print('snapshot contents:', sorted(os.listdir(model_dir)))
    import glob
    w = glob.glob(os.path.join(model_dir, '**', 'model.safetensors'), recursive=True)
    print('model.safetensors found:', w,
          [os.path.getsize(p) for p in w] if w else '')
    cfg = os.path.join(model_dir, 'config.json')
    if os.path.exists(cfg):
        print('config.json:', open(cfg).read()[:400])

import laya
agent = None
loader_used = None
errors = {}
if model_dir and hasattr(laya, 'load'):
    try:
        agent = laya.load(model_dir)
        loader_used = 'laya.load(local PIXMELT snapshot dir)'
        print('LOADED via', loader_used)
    except Exception as e:
        errors['laya.load'] = repr(e)
        print('laya.load FAILED:', repr(e))
if agent is None and model_dir and hasattr(laya, 'Agent'):
    for kwargs in ({'vegaml': False}, {}):
        try:
            agent = laya.Agent(model_dir, **kwargs)
            loader_used = f'laya.Agent(local dir, {kwargs})'
            print('LOADED via', loader_used)
            break
        except Exception as e:
            errors[f'laya.Agent(local, {kwargs})'] = repr(e)
            print(f'laya.Agent(local dir, {kwargs}) FAILED:', repr(e))
if agent is None and hasattr(laya, 'Agent'):
    for kwargs in ({'vegaml': False}, {}):
        try:
            agent = laya.Agent('convaiinnovations/laya', **kwargs)
            loader_used = f'laya.Agent(convaiinnovations/laya, {kwargs})'
            print('LOADED via', loader_used, '(upstream named checkpoint)')
            break
        except Exception as e:
            errors[f'laya.Agent(upstream, {kwargs})'] = repr(e)
            print(f'laya.Agent(convaiinnovations/laya, {kwargs}) FAILED:', repr(e))
if agent is None:
    raise RuntimeError(
        'NO classic Laya checkpoint could be loaded. Failures: ' + json.dumps(errors, indent=1)
        + '\nDo NOT continue: running the test without a loaded classic checkpoint would '
          'produce fake scores. Fix the loader (pin an older laya version matching the '
          'snapshot, or load DecisionModel directly from the snapshot rl_common.py) and re-run.')

print()
print('agent object:', type(agent))
print('loader used:', loader_used)
# Guard: refuse a Vega-engine load. The classic checkpoint is ModernBERT
# (architectures LayaTypedDecisions). If the loaded agent reports a Vega
# reader, stop here — that is a different model.
agent_repr = repr(agent) + str(getattr(agent, 'cfg', '')) + str(getattr(agent, 'config', ''))
if 'vega' in agent_repr.lower() and 'LayaTypedDecisions' not in agent_repr:
    raise RuntimeError('Loaded agent looks like the Vega engine, not the classic '
                       'ModernBERT checkpoint. Stopping — wrong model.')
print('classic-checkpoint guard: passed (no Vega-only load detected)')
try:
    import torch
    print('torch cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('device: GPU', torch.cuda.get_device_name(0))
    else:
        print('device: CPU')
except Exception as e:
    print('device check failed:', e)
load_secs = time.perf_counter() - t0
print(f'load time: {load_secs:.1f} s')
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f'RAM at load: total {vm.total/1e9:.1f} GB, available {vm.available/1e9:.1f} GB, used {vm.used/1e9:.1f} GB')
except Exception:
    pass
LOAD_INFO = {'loader_used': loader_used, 'load_seconds': round(load_secs, 1),
             'errors_before_success': errors}
print('LOAD_INFO =', LOAD_INFO)


In [ ]:
# Cell 5 — upload the v3 decision graph and verify it.
# Upload laya-crpc-bns-section-decision-tree-v3.json when the file picker
# appears. The graph is the source of truth for section ids, node
# questions/options and the root issue->section map.
# v3 (Lane A): 34 sections / 48 nodes / 79 edges. The +2/+2/+2 over v2
# (32/46/77) are the two new sections hma-9 (HMA s.9, restitution of
# conjugal rights) and pwdva-22 (PWDVA s.22, compensation orders) and their
# leaf nodes/edges. The D2 patch alone left counts at 32/46/77.
from google.colab import files
import hashlib
import json
print('Upload laya-crpc-bns-section-decision-tree-v3.json now (file picker).')
uploaded = files.upload()
print('uploaded:', list(uploaded.keys()))
GRAPH_FILE = 'laya-crpc-bns-section-decision-tree-v3.json'
assert GRAPH_FILE in uploaded, f'{GRAPH_FILE} was not uploaded — upload it and re-run this cell.'
GRAPH_SHA = hashlib.sha256(uploaded[GRAPH_FILE]).hexdigest()
print('graph SHA-256:', GRAPH_SHA)
assert GRAPH_SHA == '67bf2b04ca946f7c7fa3062068e01ce051fbce8f4d740c281afe6fff841c34a5', (
    'Graph SHA mismatch — the v3 file has changed or the wrong file was uploaded. '
    'Expected 67bf2b04ca946f7c7fa3062068e01ce051fbce8f4d740c281afe6fff841c34a5. STOP.')
GRAPH = json.loads(uploaded[GRAPH_FILE].decode('utf-8'))
n_sections = len(GRAPH['sections'])
n_nodes = len(GRAPH['nodes'])
n_edges = len(GRAPH['edges'])
print(f'verified: {n_sections} sections, {n_nodes} nodes, {n_edges} edges')
assert (n_sections, n_nodes, n_edges) == (34, 48, 79), (
    f'Graph counts do not match v3 (expected 34/48/79, got {n_sections}/{n_nodes}/{n_edges}). '
    'You may have uploaded the v2 graph — upload the v3 file and re-run.')
NODES = {n['node_id']: n for n in GRAPH['nodes']}
SECTION_IDS = {s['id'] for s in GRAPH['sections']}
print('graph OK — 34 sections / 48 nodes / 79 edges. Root node:',
      NODES['root:issue-classification']['question'][:80], '...')
# KNOWN PRE-EXISTING v2 DEFECTS, carried into v3 byte-identical (Lane A):
# 3 edges point at non-nodes (escalate_clarify, bns-316, escalate_demand_first)
# and bnss-144 related_sections carries the wrong id 'hmga-6' (real id
# 'hmga-1956-s6'). Out of scope, intentionally not fixed. NO whole-file
# "no dangling references" assert is added here on purpose: it would fail the
# run on these pre-existing defects. The counts + SHA asserts above are the
# only graph checks in this cell.


In [ ]:
# Cell 5b — upload the FROZEN brutal bank and verify it (read-only).
# Upload brutal-bank-v1-frozen.json when the file picker appears.
# The file is NEVER written to by this notebook (no write, no edit, no
# re-freeze). Verified BEFORE any batch runs: SHA-256 must equal
# 934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717 and the file
# must hold exactly 230 questions (B-001..B-230). Either check failing stops
# the run with an error — do not continue past this cell.
print('Upload brutal-bank-v1-frozen.json now (file picker).')
_uploaded_bank = files.upload()
print('uploaded:', list(_uploaded_bank.keys()))
BANK_FILE = 'brutal-bank-v1-frozen.json'
assert BANK_FILE in _uploaded_bank, f'{BANK_FILE} was not uploaded — upload it and re-run this cell.'
_bank_bytes = _uploaded_bank[BANK_FILE]
_bank_sha = hashlib.sha256(_bank_bytes).hexdigest()
print('bank SHA-256:', _bank_sha)
assert _bank_sha == '934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717', (
    'Bank SHA mismatch — the frozen bank file has changed or the wrong file was uploaded. STOP.')
BANK = json.loads(_bank_bytes.decode('utf-8'))
BANK_QUESTIONS = BANK['questions']
assert len(BANK_QUESTIONS) == 230, f'Bank must hold 230 questions, got {len(BANK_QUESTIONS)}. STOP.'
BANK_BY_ID = {q['id']: q for q in BANK_QUESTIONS}
assert [q['id'] for q in BANK_QUESTIONS] == [f'B-{i:03d}' for i in range(1, 231)], 'Bank IDs are not B-001..B-230 in order. STOP.'
BRUTAL_RESULTS = []  # every batch cell appends its records here, in order
print('bank OK: 230 questions (B-001..B-230), SHA verified. NEVER edit this file.')
print('Batches are sequential slices (B1=B-001..B-025, ..., B10=B-226..B-230),')
print('NOT stratified across categories (plan section 4.2) — see build note; a crash loses whole categories.')


In [ ]:
# Cell 6 — the 16-question test set + CORRECTED ground truth (v2, D1-A).
# Questions and facts are copied from ground-truth-v2-hma27-A.json (frozen by
# the rater lane 2026-11-27). "facts" are the question's stated facts as
# annotated in the backtest (proceeding pending, decree, divorce asked,
# demand made, violence stated) — fixed annotations, never inferred
# from keywords at run time.
# v4 ADDS one facts field, "residence_claim" (bool, frozen annotation, same
# status as the other facts): true iff the question states eviction /
# lockout / being thrown out / changed lock / cannot enter the house, OR a
# claim to stay in the shared house, OR a threat to throw the person out.
# "I want my son/daughter to stay with me" is a CUSTODY stay, not a claim
# to stay in the shared house, so H5/H14 are False under this rule.
# v6 D1-A (rater decision 2026-11-27 ~09:46 IST): HMA 27 REMOVED from K0/H3/H10/H15
# ground truth, no replacement label (GT 57 -> 53 sections). Every other
# question, section label, court, escalate expectation, fact annotation and
# question wording is byte-identical to v1. The D2 gate (Cell 7) ships in the
# same change; D1 without D2 (or D2 without D1) leaves K0/H10 non-exact.
# NEW frozen fact "being_filed" (D2): True iff the user states they are filing,
# will file, or have instructed the filing of an HMA petition. Wanting a divorce
# (H5 "I want a divorce") does NOT count. All 16 are False here — no question
# states a filing; H14 states a case "already going on" (proceeding_pending),
# not a fresh filing. The gate reads it via facts.get('being_filed', False).
# NEW field "gt_escalation_kinds" per question (v2 GT, Lane 2 typed kinds;
# 'fact' never appears here — it is informational, not a scoring kind).
# LABEL_BY_ID gains the two graph-v3 sections (hma-9, pwdva-22) so kept ids
# always have a display label; the pipeline cannot emit them yet (Cell 7 note 11).
TEST_SET = [
 {
  "id": "K0",
  "kind": "known (2026-11-26 retest, rephrased)",
  "question": "My husband left us two years ago and gives no money. I earn forty thousand a month, it is not enough for me and my eight year old daughter. My gold worth five lakh and two lakh cash my parents gave me is with my husband's family and they will not return it. They are throwing me out of the house which is in my husband's name. Who will get my daughter, how do I get my gold back, and which court do I go to?",
  "gt_labels": [
   "BNSS 144",
   "PWDVA 20",
   "GWA 1890",
   "HMGA s.6",
   "WELFARE",
   "BNS 316",
   "PWDVA 17",
   "PWDVA 18",
   "PWDVA 19"
  ],
  "gt_ids": [
   "bnss-144",
   "pwdva-20",
   "gwa-1890",
   "hmga-1956-s6",
   "welfare-principle",
   "bns-316",
   "pwdva-17",
   "pwdva-18",
   "pwdva-19"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": True,
   "violence": False,
   "residence_claim": True,
   "money_fact": True,
   "gold_fact": True
  },
  "court_expected": [
   "Family Court",
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "quantum",
   "custody",
   "residence",
   "recovery"
  ]
 },
 {
  "id": "H1",
  "kind": "held-out, single-issue maintenance",
  "question": "My husband has not given me any money for three months. I earn fifteen thousand a month and I have two small children and no case filed anywhere.",
  "gt_labels": [
   "BNSS 144",
   "PWDVA 20"
  ],
  "gt_ids": [
   "bnss-144",
   "pwdva-20"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": None,
   "violence": False,
   "residence_claim": False,
   "money_fact": True,
   "gold_fact": False
  },
  "court_expected": [
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "quantum"
  ]
 },
 {
  "id": "H2",
  "kind": "held-out, single-issue custody, no proceeding",
  "question": "Who will get my six year old son? My husband says as the father he will take him away from me.",
  "gt_labels": [
   "GWA 1890",
   "HMGA s.6",
   "WELFARE"
  ],
  "gt_ids": [
   "gwa-1890",
   "hmga-1956-s6",
   "welfare-principle"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": None,
   "violence": False,
   "residence_claim": False,
   "money_fact": False,
   "gold_fact": False
  },
  "court_expected": [
   "Family Court"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "custody"
  ]
 },
 {
  "id": "H3",
  "kind": "held-out, single-issue Stridhan, demand not yet made",
  "question": "My mother gave me gold at my wedding and my in-laws kept it. I have never asked them to return it.",
  "gt_labels": [
   "BNS 316"
  ],
  "gt_ids": [
   "bns-316"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": False,
   "violence": False,
   "residence_claim": False,
   "money_fact": False,
   "gold_fact": True
  },
  "court_expected": [
   "Family Court",
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "demand"
  ]
 },
 {
  "id": "H4",
  "kind": "held-out, single-issue residence",
  "question": "They changed the lock and I cannot enter my house. The house is in my husband's name.",
  "gt_labels": [
   "PWDVA 17",
   "PWDVA 18",
   "PWDVA 19"
  ],
  "gt_ids": [
   "pwdva-17",
   "pwdva-18",
   "pwdva-19"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": None,
   "violence": False,
   "residence_claim": True,
   "money_fact": False,
   "gold_fact": False
  },
  "court_expected": [
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "residence"
  ]
 },
 {
  "id": "H5",
  "kind": "held-out, multi-issue, divorce asked",
  "question": "I want a divorce from my husband. He beats me, I need money every month and I want my daughter to stay with me.",
  "gt_labels": [
   "HMA 13",
   "BNSS 144",
   "PWDVA 20",
   "PWDVA 18",
   "GWA 1890",
   "HMGA s.6",
   "WELFARE"
  ],
  "gt_ids": [
   "hma-13",
   "bnss-144",
   "pwdva-20",
   "pwdva-18",
   "gwa-1890",
   "hmga-1956-s6",
   "welfare-principle"
  ],
  "facts": {
   "divorce_asked": True,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": None,
   "violence": True,
   "residence_claim": False,
   "money_fact": True,
   "gold_fact": False
  },
  "court_expected": [
   "Family Court",
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "quantum",
   "custody"
  ]
 },
 {
  "id": "H6",
  "kind": "held-out, divorce NOT asked (explicit)",
  "question": "I do not want a divorce. I just want money every month from my husband and to stay in the house with my daughter.",
  "gt_labels": [
   "BNSS 144",
   "PWDVA 20",
   "PWDVA 17",
   "PWDVA 18",
   "PWDVA 19"
  ],
  "gt_ids": [
   "bnss-144",
   "pwdva-20",
   "pwdva-17",
   "pwdva-18",
   "pwdva-19"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": None,
   "violence": False,
   "residence_claim": True,
   "money_fact": True,
   "gold_fact": False
  },
  "court_expected": [
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "quantum",
   "residence"
  ]
 },
 {
  "id": "H7",
  "kind": "held-out, birth certificate",
  "question": "How do I get my daughter's birth certificate? The hospital did not give us one.",
  "gt_labels": [
   "Births and Deaths Act 1969"
  ],
  "gt_ids": [
   "births-deaths-act-1969"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": None,
   "violence": False,
   "residence_claim": False,
   "money_fact": False,
   "gold_fact": False
  },
  "court_expected": [
   "Registrar"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "procedure"
  ]
 },
 {
  "id": "H8",
  "kind": "held-out, salary slips missing, income unknown",
  "question": "I need maintenance but I do not have my husband's salary slips and I do not know what he earns.",
  "gt_labels": [
   "BNSS 144",
   "PWDVA 20"
  ],
  "gt_ids": [
   "bnss-144",
   "pwdva-20"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": None,
   "violence": False,
   "residence_claim": False,
   "money_fact": True,
   "gold_fact": False
  },
  "court_expected": [
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "quantum"
  ]
 },
 {
  "id": "H9",
  "kind": "held-out, income unknown, quantum asked",
  "question": "How much money can I get every month from my husband?",
  "gt_labels": [
   "BNSS 144",
   "PWDVA 20"
  ],
  "gt_ids": [
   "bnss-144",
   "pwdva-20"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": None,
   "violence": False,
   "residence_claim": False,
   "money_fact": True,
   "gold_fact": False
  },
  "court_expected": [
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "quantum"
  ]
 },
 {
  "id": "H10",
  "kind": "held-out, demand not yet made + maintenance",
  "question": "My husband has my gold and I have not asked for it back yet, and he gives no money for me and my child.",
  "gt_labels": [
   "BNS 316",
   "BNSS 144",
   "PWDVA 20"
  ],
  "gt_ids": [
   "bns-316",
   "bnss-144",
   "pwdva-20"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": False,
   "violence": False,
   "residence_claim": False,
   "money_fact": True,
   "gold_fact": True
  },
  "court_expected": [
   "Family Court",
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "demand",
   "quantum"
  ]
 },
 {
  "id": "H11",
  "kind": "held-out, no proceeding pending, interim money asked",
  "question": "I need money while my case is going on, but I have not filed any case yet.",
  "gt_labels": [
   "BNSS 144",
   "PWDVA 20"
  ],
  "gt_ids": [
   "bnss-144",
   "pwdva-20"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": None,
   "violence": False,
   "residence_claim": False,
   "money_fact": True,
   "gold_fact": False
  },
  "court_expected": [
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "quantum"
  ]
 },
 {
  "id": "H12",
  "kind": "held-out, out of scope",
  "question": "My brother and I are fighting over our father's ancestral house. Who owns it and can I stop him selling it?",
  "gt_labels": [],
  "gt_ids": [],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": None,
   "violence": False,
   "residence_claim": False,
   "money_fact": False,
   "gold_fact": False
  },
  "court_expected": [],
  "escalate_expected": False,
  "gt_escalation_kinds": []
 },
 {
  "id": "H13",
  "kind": "held-out, multi-issue violence + residence + school fees",
  "question": "He beats me, throws me out of the house and gives no money for my son's school fees.",
  "gt_labels": [
   "PWDVA 18",
   "PWDVA 17",
   "PWDVA 19",
   "PWDVA 20",
   "BNSS 144"
  ],
  "gt_ids": [
   "pwdva-18",
   "pwdva-17",
   "pwdva-19",
   "pwdva-20",
   "bnss-144"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": None,
   "violence": True,
   "residence_claim": True,
   "money_fact": True,
   "gold_fact": False
  },
  "court_expected": [
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "quantum",
   "residence"
  ]
 },
 {
  "id": "H14",
  "kind": "held-out, proceeding pending (divorce case on in Family Court)",
  "question": "My divorce case is already going on in the Family Court. I want my son to stay with me and money for his school fees while the case goes on.",
  "gt_labels": [
   "HMA 13",
   "HMA 26",
   "HMGA s.6",
   "WELFARE",
   "HMA 24",
   "BNSS 144",
   "PWDVA 20"
  ],
  "gt_ids": [
   "hma-13",
   "hma-26",
   "hmga-1956-s6",
   "welfare-principle",
   "hma-24",
   "bnss-144",
   "pwdva-20"
  ],
  "facts": {
   "divorce_asked": True,
   "proceeding_pending": True,
   "being_filed": False,
   "decree": False,
   "demand_made": None,
   "violence": False,
   "residence_claim": False,
   "money_fact": True,
   "gold_fact": False
  },
  "court_expected": [
   "Family Court",
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "quantum",
   "custody"
  ]
 },
 {
  "id": "H15",
  "kind": "held-out, Stridhan sold",
  "question": "He sold my gold without asking me. The gold was given to me by my parents at my wedding.",
  "gt_labels": [
   "BNS 316"
  ],
  "gt_ids": [
   "bns-316"
  ],
  "facts": {
   "divorce_asked": False,
   "proceeding_pending": False,
   "being_filed": False,
   "decree": False,
   "demand_made": True,
   "violence": False,
   "residence_claim": False,
   "money_fact": False,
   "gold_fact": True
  },
  "court_expected": [
   "Family Court",
   "Magistrate"
  ],
  "escalate_expected": True,
  "gt_escalation_kinds": [
   "recovery"
  ]
 }
]

LABEL_BY_ID = {
 "bnss-144": "BNSS 144",
 "hma-24": "HMA 24",
 "hma-25": "HMA 25",
 "pwdva-20": "PWDVA 20",
 "pwdva-22": "PWDVA 22",
 "hma-26": "HMA 26",
 "gwa-1890": "GWA 1890",
 "hmga-1956-s6": "HMGA s.6",
 "welfare-principle": "WELFARE",
 "hma-27": "HMA 27",
 "bns-316": "BNS 316",
 "pwdva-17": "PWDVA 17",
 "pwdva-18": "PWDVA 18",
 "pwdva-19": "PWDVA 19",
 "hma-13": "HMA 13",
 "hma-13b": "HMA 13B",
 "hma-9": "HMA 9",
 "births-deaths-act-1969": "Births and Deaths Act 1969"
}

print(f'test set loaded: {len(TEST_SET)} questions')
for t in TEST_SET:
    print(f"  {t['id']:3s} [{t['kind']}] GT={t['gt_labels']}")
assert [t['id'] for t in TEST_SET] == ['K0'] + [f'H{i}' for i in range(1, 16)], 'test set IDs are not K0 + H1-H15'
assert all(set(t['gt_ids']) <= set(LABEL_BY_ID) for t in TEST_SET), 'GT id with no label'
assert sum(len(t['gt_ids']) for t in TEST_SET) == 53, 'v2 GT must total 53 sections (D1-A)'
assert all('being_filed' in t['facts'] for t in TEST_SET), 'being_filed missing on some question'
assert all(t['facts']['being_filed'] is False for t in TEST_SET), 'being_filed must be False on all 16 (no question states a filing)'
assert all('hma-27' not in t['gt_ids'] for t in TEST_SET if t['id'] in ('K0', 'H3', 'H10', 'H15')), 'D1-A: hma-27 must be out of K0/H3/H10/H15 GT'
print('test set OK: 16 questions (K0 + H1-H15), corrected v2 ground truth (53 sections, D1-A).')


In [ ]:
# Cell 7 — v6 (D1-A + D2 gate + Lane 2 typed kinds; v5 fixes kept) LIVE Stage 1 + Stage 2.
# SCOPE: residence precision ONLY, on top of v3. v3 live (2026-11-26):
# exact 5/16, mean P/R 0.673/0.866, micro P/R 0.671/0.895, TP/FP/FN
# 51/25/6, court 11/16, escalates 14/16. Main leak: residence R1 fired
# 11/16 (R0: 5/16) — H2/H10/H12 residence FPs, H13 split, H6 miss.
# v4 changes (see v4-change-log.md for the full diff + predictions):
#  (a) v3 thresholds KEPT (M 0.10, C 0.20, R 0.15,
#      divorce-asked 0.45 / divorce-pending 0.12);
#  (b) residence fires on the R0 prompt (exact v1 text) at 0.15, R1 recorded
#      for diagnosis only and never fires;
#  (c) NEW Cell 6 facts field residence_claim + Stage 1 fact gate
#      (claim False -> residence final forced to 0.0, never fires);
#  (d) Stage 2 residence keeps PWDVA 17+18+19 as a full set, never split.
# No live Colab run here — this builder only writes the notebook.
#
# WHAT CHANGED vs v1 (Cell 7):
#  1. CLAUSE-SPLIT (plan §2 Step 2, rule implemented verbatim): split on
#     sentence boundaries [.!?;] then on "and"/"but" ONLY when followed by
#     I|he|they|my husband|my son|my daughter (so "gold and cash" and "money
#     and my child" do NOT split); NEVER split inside a negation (a fragment
#     whose first 4 words contain not / n't / never / cannot stays attached —
#     "they will not return it", "have not filed any case yet", "cannot enter
#     my house"); drop clauses with <3 words. Full question kept as baseline
#     (stage1_raw) and included in the max pool as the fallback. Per-issue
#     score = MAX over clauses (+ full fallback), NEVER mean. fired_clause =
#     argmax clause text; fired_source = which source won (clause/full/aux).
#     KNOWN DEVIATION, documented not hidden: the plan's worked splits also
#     split at bare commas ("He beats me, throws me out...", "Who will get my
#     daughter, how do I get my gold back..."). The verbatim rule has no
#     comma-only split, so H13 stays a single clause and H5/K0 keep some
#     comma-joined clauses whole. rec['clauses'] records the actual split per
#     question, so this is auditable in the results file.
#  2. AUX PROMPTS (plan §2 Step 3): the 12 verbatim phrases as auxiliary
#     noul prompts, each 'Does this question say "<verbatim phrase>" or mean
#     the same thing?', run on the FULL question. Issue score =
#     max(main prompt, aux prompts). Includes the H4 lock, H8 salary-slips,
#     H11 not-filed-yet, H14 pending-case + son-custody + school-fees phrases.
#  3. PER-ISSUE THRESHOLDS — Lane A PRIMARY vector (v4), NEVER a single
#     global threshold. maintenance 0.10 (v3, kept), custody 0.20 (v3, kept),
#     stridhan 0.50 (keep), residence 0.15 R0 (keep the value, change the
#     prompt), violence 0.50 (keep), divorce-asked 0.45 (v3, kept),
#     divorce-pending 0.12 (v3, kept), birth_certificate 0.50 (keep),
#     in_scope 0.50 (no plan range; default).
#     Divorce IS split into two signals (asked vs pending) as the plan
#     requires: divorce fires if asked-final >= 0.45 OR pending-final >= 0.12.
#     WARNING, recorded before the live run (see v4-change-log.md §5):
#     applying this PRIMARY vector to the v3 live finals newly fires
#     maintenance on H2/H4/H12 and custody on H7/H10/H12 — those are
#     PREDICTED new FPs. The residence-only counterfactual (v3 thresholds
#     kept) predicts 10/16 exact; the full PRIMARY vector predicts 6/16.
#     The vector is applied as specified; the numbers decide in Colab.
#  4. RESIDENCE R0 FIRES (v4; v3 fired R1). R0 = the exact v1 prompt:
#     "Does this question ask about staying in, or being thrown out of, the
#     shared house, or about protection from threats or violence at home?"
#     R1 = residence-only: "Does this question ask about staying in, or being
#     thrown out of, the shared house or home?" Both probs recorded
#     (residence_r0 / residence_r1, full and clause-max) plus the aux
#     residence max; the R0 final = max(R0 full, R0 clause-max, aux).
#     RESIDENCE_PROMPT="R0" selects the firing prompt; R1 is retained for
#     comparison only and never fires. R0 alone does NOT fix H2: its R0
#     clause-max is 0.8464 ("take him away" read as a threat). The fact
#     gate (note 8) is what blocks H2/H3/H5/H10/H12/H14.
#  5. HMA 25 PRE-GENERATION GATE (plan §3.1) — FITTED, both layers:
#     Layer A filters the option 'permanent alimony at/after an HMA decree'
#     out of maintenance:maintenance-root BEFORE offering it to Laya when
#     facts['decree'] is False (recorded in gated_out). Layer B in stage2():
#     hma-25 is added ONLY if facts['decree'] is True, otherwise it goes to
#     gated_out, NOT to dropped[] (it was never a candidate). The old
#     add-then-drop recheck block for hma-25 is DELETED. Consequence, stated
#     in Cell 8 too: forbidden_proposed_before_gate is 0-by-construction for
#     the HMA 25 route after Layer A (the option cannot be picked); the honest
#     check is the offline shadow test below (PASS printed every run) plus
#     kept_ids for H5/H9 byte-identical to the v1 run.
#     NOT fitted in this lane (plan §3.2, out of scope — stated, not hidden):
#     HMA 24 / HMA 26 / divorce-gate / custody-route pre-generation gating.
#     The residence fact gate (note 8) IS fitted in v4 — it needed the new
#     Cell 6 facts field residence_claim, added as a frozen annotation.
#     HMA 24 / HMA 26 still use the v1 add-then-drop recheck shape inside
#     stage2().
#  6. COURT FALLBACK (plan §3.3): court_for(kept, fired, probs, facts,
#     in_scope). Kept non-empty: as v1, except Registrar is now membership
#     ('births-deaths-act-1969' in kept), not equality. Kept empty + in-scope:
#     NO guessed court and never [] — court is ["ESCALATE — no section kept,
#     court cannot be derived"] + top-2 candidate issues as payload + a typed
#     court escalation. A provisional court is emitted ONLY if top prob >= 0.40
#     AND margin over second >= 0.15 — that condition holds for NONE of the 5
#     empty questions, so on this set the fallback is escalation-only.
#     Kept empty + out-of-scope: [] / escalate False (H12 must stay empty).
#     in_scope comes from an 8th Stage 1 noul question: "Does this question
#     concern a Hindu matrimonial / domestic-relationship family-law matter?"
#  7. TYPED ESCALATIONS (plan §3.4): escalations_typed entries {kind, text},
#     kinds quantum / court / demand / fact. Quantum ONLY for maintenance
#     questions (maintenance fired OR a maintenance section kept), never an
#     amount, ask-income text, never demand salary slips. Demand-first
#     Stridhan escalation typed as demand (kept, BNS 316 NOT gated — plan
#     §3.2: deliberately propose-and-escalate). Plain-text escalations list is
#     ALSO kept (backward compatible); Cell 8 scores from the typed
#     quantum/court flag, NOT from bool(escalations) — see Cell 8 header.
#     KNOWN CONSEQUENCE (flagged, not hidden): under typed quantum/court
#     scoring, H2/H3/H7/H15 have no quantum/court reason (H2 custody+residence,
#     H3 demand-only, H7 birth-certificate, H15 stridhan-only with demand
#     already made), so they read escalates=False against expected True.
#     H7 is the plan's flagged OPEN ambiguity (fully answerable question with
#     escalate_expected=True and no named reason). Cell 8 also reports the
#     any-typed and bool-plain diagnostics so this effect is visible, not
#     buried. Do NOT "fix" it by re-attaching the v1 boilerplate.
#  8. v4 RESIDENCE FACT GATE + FULL SET (this version's focused fix):
#     residence_claim (Cell 6, frozen annotation) is true iff the question
#     states eviction / lockout / being thrown out / changed lock / cannot
#     enter the house, OR a claim to stay in the shared house, OR a threat
#     to throw the person out. Values: K0/H4/H6/H13 True, all others False.
#     "I want my son/daughter to stay with me" (H5, H14) is a custody stay,
#     NOT a claim to stay in the shared house, so both are False under the
#     rule as written — no value contradicts the rule. Gate: claim False
#     forces residence final to 0.0 before firing (recorded in gated_out);
#     claim True fires per the R0 score. Stage 2 then keeps PWDVA 17+18+19
#     as a full set when residence fires; the sub-choice outcome is recorded
#     in rec['residence_sub_choice_info'] for information only and never
#     splits the set. The violence -> PWDVA 18 path is COMPLETELY UNTOUCHED:
#     when residence is gated out and violence fires, PWDVA 18 still comes
#     in through violence (that is how H5's missing PWDVA 18 is predicted
#     to return — see v4-change-log.md §5).
#  9. v6 D2 HMA 27 GATE (drop, paired with D1-A): in stage2(), after the HMA 25
#     Layer B block — if 'hma-27' is in kept and neither proceeding_pending nor
#     being_filed is True, hma-27 is REMOVED from kept and recorded in gated_out
#     with the D2 reason (gated-before-generation, like HMA 25 Layer B; never
#     added to dropped[]). BNS 316 is NOT gated. hma-27 stays in
#     candidates_before_gate. The graph v3 rule R-HMA27-04 is if_fail=drop with
#     the same wording. Verified by the offline shadow test below (PASS printed
#     every run). "Being filed" = the user states they are filing / will file /
#     have instructed the filing of an HMA petition; wanting a divorce does not
#     count (Cell 6 note).
# 10. v6 TYPED ESCALATIONS (Lane 2 section 4.1 + D1-A recovery text): stage2()
#     now also emits custody (kept GWA/HMA-26), residence ('residence' in fired,
#     post-gate — NOT kept PWDVA 17/18/19, so H5's violence-only PWDVA 18 does
#     not get residence text), recovery (stridhan kept + demand_made True, with
#     the D1-A civil-route text: Family Court under FCA s.7(1) Expl. (c) /
#     CPC s.151 unless an HMA proceeding is pending/being filed), procedure
#     (births-deaths kept), and safety (PWDVA 18 kept — see the in-code note
#     for why this trigger, not the Lane 2 violence-fact trigger, is used).
#     D5 rater confirmation of the H2/H4 kinds is still outstanding.
#     The 'fact' kind stays informational and is never in the scoring set.
# 11. v6 GRAPH v3 LABELS (hma-9 / pwdva-22): the labels exist in the graph and
#     in LABEL_BY_ID, but NO Stage 1 issue maps to them and stage2() has no
#     route that emits them (graph v3 adds only stage2 leaf nodes). B-018
#     (hma-9) and B-019 (pwdva-22) are therefore GUARANTEED MISSES in this
#     build. Emitting them needs a new Stage 1 issue prompt + stage2 route —
#     a pipeline change, not made here (no-logic-change constraint).
# 12. v6 FACT DERIVATION (STOPGAP — derive_facts_from_text / complete_facts
#     below): the brutal bank carries NO fact annotations and the sealed sets
#     carry only five (divorce_asked / proceeding_pending / decree /
#     demand_made / violence), while the pipeline needs a full facts dict.
#     Facts for those questions are derived from the question TEXT by the
#     frozen rules in that helper. Deterministic, identical for every
#     question, never tuned per question, never read from GT. Known limits,
#     recorded not hidden: (a) Kannada-script questions match no English
#     pattern and fall back to defaults (proceeding False, demand None,
#     violence False, residence False, money/gold False) — the v5 gates then
#     veto almost everything, so Kannada scores through this build measure
#     the English-text gates, not the model; model-extracted facts (v6
#     deep-dive section 2) are the proper fix. (b) Money need phrased without
#     money words ("books and uniforms") reads money_fact=False. (c) Gold
#     mentioned as an object, not a claim, reads gold_fact=True — the gate
#     then does not veto and the model score decides (C9 is the stress axis).
# 13. v6 REFACTOR: the per-question Stage 1 + Stage 2 body is ONE reusable
#     function run_one_question(t) (below). The body is the v5 loop body,
#     re-indented only — no threshold, prompt, scoring-logic or fact change
#     (the builder verifies this by a dedent-equivalence check). The TEST_SET
#     loop, the batch cells and the sealed cells all call it. t must carry
#     'id', 'question', 'facts', 'gt_ids', 'gt_labels'.
# HARD RULES (unchanged from v1): every issue pick comes from agent.predict().
# If a predict call raises on the FULL-question pass, the question is recorded
# as an ERROR and scored as a miss. There is NO fallback to keyword / phrase /
# rule matching. Clause/aux call failures are recorded per clause and that
# clause is skipped (partial scores stand); they never invent a score.
# Sub-choices run ONLY for fired issues, options from the uploaded v2 graph,
# body = the issue's argmax clause (full question if that clause has <5 words).
import re
import time

# --- Recommended per-issue threshold vector (plan §2 Step 1) ---
THRESHOLDS = {
    'maintenance': 0.10,
    'custody': 0.20,
    'stridhan': 0.50,
    'residence': 0.15,
    'violence': 0.50,
    'divorce_asked': 0.45,
    'divorce_pending': 0.12,
    'birth_certificate': 0.50,
    'in_scope': 0.50,
}
# NEVER a single global threshold: true-positive floors differ by 0.82 across
# issues (maintenance 0.0773 vs stridhan 0.8929) and residence/custody/divorce
# ranges are inverted/overlapping (plan §2 Step 1, "Why a single global
# threshold cannot work"). Keep this dict explicit; change one value at a time.

RESIDENCE_R0 = 'Does this question ask about staying in, or being thrown out of, the shared house, or about protection from threats or violence at home?'
RESIDENCE_R1 = 'Does this question ask about staying in, or being thrown out of, the shared house or home?'
RESIDENCE_PROMPT = 'R0'  # v4: R0 fires; set to 'R1' only for a comparison run
DIVORCE_PENDING_Q = 'Does this question say a divorce case is already going on or pending in court?'
IN_SCOPE_Q = 'Does this question concern a Hindu matrimonial / domestic-relationship family-law matter?'

ISSUES = [
    ('maintenance', 'Does this question ask for monthly money or maintenance for the wife or a child?'),
    ('custody', 'Does this question ask who will get or look after the child, or about guardianship or custody?'),
    ('stridhan', 'Does this question ask about gold, cash or gifts given to the wife at marriage that are held by the husband or his family?'),
    ('residence', RESIDENCE_R0),
    ('violence', 'Does this question say the husband beats, threatens or harasses the wife?'),
    ('divorce', 'Does this question ask for a divorce, or for ending the marriage?'),
    ('birth_certificate', "Does this question ask how to get a child's birth certificate?"),
]
ISSUE_NODE = {
    'maintenance': 'maintenance:maintenance-root',
    'custody': 'custody:custody_sub_issue_choice',
    'stridhan': 'stridhan:stridhan-subissue',
    'residence': 'residence:residence_sub_issue',
    'divorce': 'procedure:divorce-gate',
}
ISSUE_EXTRA_NODE = {
    'stridhan': 'stridhan:civil-vs-criminal',
    'residence': 'residence:protection_vs_residence',
}

# --- 12 auxiliary noul prompts, verbatim phrases from plan §2 Step 3 ---
# Each becomes: Does this question say "<phrase>" or mean the same thing?
AUX_PROMPTS = [
    {'issue': 'residence', 'phrase': 'They changed the lock and I cannot enter my house'},
    {'issue': 'maintenance', 'phrase': 'I need money while my case is going on, but I have not filed any case yet'},
    {'issue': 'maintenance', 'phrase': "I need maintenance but I do not have my husband's salary slips and I do not know what he earns"},
    {'issue': 'maintenance', 'phrase': "gives no money for my son's school fees"},
    {'issue': 'maintenance', 'phrase': 'money for his school fees while the case goes on'},
    {'issue': 'maintenance', 'phrase': 'he gives no money for me and my child'},
    {'issue': 'maintenance', 'phrase': 'My husband has not given me any money for three months'},
    {'issue': 'custody', 'phrase': 'I want my son to stay with me'},
    {'issue': 'residence', 'phrase': 'throws me out of the house'},
    {'issue': 'divorce_pending', 'phrase': 'My divorce case is already going on in the Family Court'},
    {'issue': 'custody', 'phrase': 'I want my daughter to stay with me'},
    {'issue': 'residence', 'phrase': 'to stay in the house with my daughter'},
]


def split_clauses(q):
    # Plan §2 Step 2 deterministic rule, verbatim (see header note 1 for the
    # one documented deviation: no comma-only split, so H13 stays one clause
    # and H5/K0 keep some comma-joined clauses whole).
    _NEG = ('not', 'never', 'cannot', "can't", "couldn't", "won't", "don't", "isn't", "wasn't")
    def _negated(fragment):
        words = fragment.split()[:4]
        return any((w.strip('.,;:').lower() in _NEG) or w.lower().endswith("n't") for w in words)
    _SPLIT_RE = re.compile(
        r'\s+and\s+(?=(?:I|he|they|my\s+(?:husband|son|daughter))\b)'
        r'|\s+but\s+(?=(?:I|he|they|my\s+(?:husband|son|daughter))\b)')
    out = []
    for sent in re.split(r'(?<=[.!?;])\s+', q.strip()):
        # Cut points: text before each kept match start, text after its end.
        # The matched " and " / " but " separator itself is dropped.
        pieces = []
        prev_end = 0
        for m in _SPLIT_RE.finditer(sent):
            if _negated(sent[m.end():]):
                continue
            pieces.append(sent[prev_end:m.start()])
            prev_end = m.end()
        pieces.append(sent[prev_end:])
        for ch in pieces:
            ch = ch.strip().strip(',;').strip()
            if len(ch.split()) >= 3:
                out.append(ch)
    return out if out else [q.strip()]


def _parse(answer, qtype):
    if isinstance(answer, dict):
        for k in ('choice', 'value', 'option', 'answer', 'probability', 'score'):
            if k in answer:
                return answer[k]
        return str(answer)
    return answer


def _ask(payload, body):
    res = agent.predict({'body': body}, payload)
    return res['answers']


def _prob_from(ans_dict, key):
    try:
        return float(_parse(ans_dict[key]['noul'], 'noul'))
    except Exception:
        return None


def stage2(fired, sub, facts):
    # Returns (kept, dropped, esc_typed, gated_out, candidates_before_gate).
    # candidates_before_gate = what Stage 1 + sub-choices proposed BEFORE the
    # HMA 25 gate (kept pre-gate union dropped section ids). HMA 25 Layer B
    # is applied here; Layer A filtered the option before offering it.
    kept_pre = set()
    dropped = []
    esc_typed = []
    gated_out = []
    if 'maintenance' in fired:
        kept_pre |= {'bnss-144', 'pwdva-20'}
        ans = sub.get('maintenance:maintenance-root', '')
        if 'interim maintenance during a pending HMA proceeding' in ans:
            kept_pre.add('hma-24')
        if 'permanent alimony at/after an HMA decree' in ans:
            kept_pre.add('hma-25')
    if 'custody' in fired:
        # v5 Fix 4 (H14 exclusive primary route): proceeding pending ->
        # HMA 26 primary; otherwise the unchanged GWA 1890 primary.
        # HMGA s.6 + WELFARE ride with either primary. The recheck
        # below (hma-26 dropped with no proceeding pending) stays as
        # Layer-B defence-in-depth; no symmetric remove-GWA recheck.
        if facts['proceeding_pending']:
            kept_pre |= {'hma-26', 'hmga-1956-s6', 'welfare-principle'}
        else:
            kept_pre |= {'gwa-1890', 'hmga-1956-s6', 'welfare-principle'}
    if 'stridhan' in fired:
        kept_pre |= {'hma-27', 'bns-316'}
        if facts['demand_made'] is False:
            esc_typed.append({'kind': 'demand', 'text': 'Stridhan: make a written demand for return first, then re-evaluate BNS s.316'})
    if 'residence' in fired:
        # v4 FULL SET: when residence fires (it already passed the fact gate
        # in Stage 1), keep PWDVA 17 + 18 + 19 as a set. The residence
        # sub-choice outcomes are recorded informationally by the caller in
        # rec['residence_sub_choice_info'] and NEVER split the set (v3 split
        # it: H13 kept only PWDVA 18, H5 kept 17+19 and lost 18).
        kept_pre |= {'pwdva-17', 'pwdva-18', 'pwdva-19'}
    if 'violence' in fired and 'residence' not in fired:
        kept_pre.add('pwdva-18')
    if 'divorce' in fired:
        gate = sub.get('procedure:divorce-gate', '')
        if facts['divorce_asked'] and 'open gate' in gate:
            kept_pre.add('hma-13')
        elif facts['divorce_asked']:
            dropped.append(('hma-13', 'divorce issue fired but the divorce-gate choice did not open the gate'))
        # HMA 13B (mutual consent) is never added: no test question states mutual consent.
    if 'birth_certificate' in fired:
        kept_pre.add('births-deaths-act-1969')
    # --- recheck / drops (v1 rules; HMA 24 / HMA 26 still add-then-drop here,
    # out of scope for this lane — see header note 5) ---
    if not facts['divorce_asked']:
        for s in ('hma-13', 'hma-13b'):
            if s in kept_pre:
                kept_pre.remove(s)
                dropped.append((s, 'divorce gate: divorce not asked'))
    if not facts['proceeding_pending'] and 'hma-24' in kept_pre:
        kept_pre.remove('hma-24')
        dropped.append(('hma-24', 'no HMA proceeding pending'))
    if not facts['proceeding_pending'] and 'hma-26' in kept_pre:
        kept_pre.remove('hma-26')
        dropped.append(('hma-26', 'no HMA proceeding pending — GWA 1890 route'))
    candidates_before_gate = sorted(kept_pre | {d[0] for d in dropped if d[0] in SECTION_IDS})
    # --- HMA 25 gate, Layer B: add ONLY if a decree exists ---
    kept = set(kept_pre)
    if 'hma-25' in kept:
        if facts['decree']:
            pass  # decree exists: keep (never exercised on this 16 — no question has decree True)
        else:
            kept.remove('hma-25')
            gated_out.append("hma-25: gated before generation (Layer B) — sub-choice said 'permanent alimony at/after an HMA decree' but decree=false")
            # NOT added to dropped[]: it was never a candidate after gating.
    # --- v6 D2 HMA 27 gate (drop, not escalate): HMA s.27 fires only when an
    # HMA proceeding (ss. 9-13/13B) is pending or being filed together with the
    # claim (D1-A corrected rule; graph v3 R-HMA27-04 if_fail=drop). If neither
    # facts['proceeding_pending'] nor facts['being_filed'] is True, hma-27 is
    # removed from kept and recorded in gated_out (gated-before-generation,
    # like HMA 25 Layer B) — never added to dropped[]. BNS 316 is NOT gated:
    # it is the criminal route and stays. hma-27 stays in
    # candidates_before_gate (computed above, before this block).
    if 'hma-27' in kept and not (facts.get('proceeding_pending') or facts.get('being_filed', False)):
        kept.remove('hma-27')
        gated_out.append("hma-27: gated (D2) — no HMA proceeding pending or being filed; HMA s.27 cannot be invoked standalone; civil return route is the Family Court property suit under FCA s.7(1) Expl. (c) / CPC s.151; BNS s.316 kept as the criminal route")
    # The old 'if not facts[decree] and hma-25 in kept: drop' recheck block is
    # DELETED per plan §3.1 (it would preserve the propose-then-drop shape).
    # --- typed escalations: quantum ONLY for maintenance questions ---
    # Income-unknown is assumed: Cell 6 has no income_known fact field (GT
    # unchanged by design), so no per-question income check is possible here.
    maint_kept = bool(kept & {'bnss-144', 'pwdva-20', 'hma-24', 'hma-25'})
    if 'maintenance' in fired or maint_kept:
        esc_typed.append({'kind': 'quantum', 'text': "Quantum: husband's income / school fees unknown — escalate the amount, never invent it"})
        esc_typed.append({'kind': 'fact', 'text': "Ask husband's income (estimate/range), school-fee amount, where husband lives now; never demand salary slips"})
    # --- v6 typed escalations: custody / residence / recovery / procedure / safety ---
    # (Lane 2 section 4.1 + D1-A recovery-text amendment; D5 rater confirmation
    # of the H2/H4 kinds still outstanding — see Cell 8 header.)
    # Triggers are kept sections and/or frozen facts ONLY — never raw scores,
    # never pre-gate fired state, never a question id. Residence uses post-gate
    # 'residence' in fired (NOT kept PWDVA 17/18/19): H5 keeps PWDVA 18 via the
    # violence-only branch and its residence text would be content-wrong there.
    if kept & {'gwa-1890', 'hma-26'}:
        _route = 'HMA s.26 (proceeding pending)' if 'hma-26' in kept else 'GWA 1890'
        esc_typed.append({'kind': 'custody', 'text': (
            f"Custody: who the child stays with is a welfare decision for the Family Court "
            f"-- the bot names the route ({_route}) but does not say who gets the child. "
            f"Escalate for interim custody and the welfare assessment. "
            f"If either parent has said they will take the child away, that needs urgent human help.")})
    if 'residence' in fired:
        esc_typed.append({'kind': 'residence', 'text': (
            "Residence: the bot names PWDVA 17-19 but cannot restore entry to the house "
            "-- a Magistrate's residence order is needed (the house being in the husband's name "
            "does not remove the right). Escalate for filing the residence application "
            "and for immediate help re-entering.")})
    if kept & {'hma-27', 'bns-316'} and facts.get('demand_made') is True:
        # v6 D1-A amendment (Lane A wording, unconditional variant — see build note):
        # the civil route is the Family Court property suit under FCA s.7(1)
        # Expl. (c) / CPC s.151 unless an HMA proceeding is pending/being filed
        # (then HMA s.27). Never cite BNS 316 alone as the civil route.
        esc_typed.append({'kind': 'recovery', 'text': (
            "Stridhan recovery: sold or withheld gold cannot be recovered or valued by the bot "
            "-- escalate to a lawyer for valuation, a police complaint under BNS s.316 "
            "and civil recovery through the Family Court under FCA s.7(1) Expl. (c) / CPC s.151 "
            "(or under HMA s.27 where an HMA proceeding is pending or being filed).")})
    if 'births-deaths-act-1969' in kept:
        esc_typed.append({'kind': 'procedure', 'text': (
            "Birth certificate: the exact procedure depends on facts not stated (date of birth, "
            "any delay, whether the hospital will issue) and requires applying in person to the "
            "Registrar. Escalate for human follow-up with the hospital / Registrar "
            "if issuance is refused or delayed.")})
    if 'pwdva-18' in kept:
        # v6 'safety' kind: trigger is PWDVA 18 KEPT, not the Lane 2 violence-fact
        # trigger. Evidence: 31/32 bank safety uses carry pwdva-18 in GT (the one
        # exception, B-108 custody-only safety, is a recorded gap); only 13/32
        # state violence in words the violence fact could see. A protection order
        # in the answer means threats/violence are live and need urgent human help.
        # On the 16 this fires on K0/H4/H5/H6/H13 only — all expected True already.
        esc_typed.append({'kind': 'safety', 'text': (
            "Safety: a protection order is in the answer -- escalate urgently for immediate "
            "safety help and a protection order under PWDVA s.18; do not treat this as information-only.")})
    return kept, dropped, esc_typed, gated_out, candidates_before_gate


def _court_for_issue(issue):
    return {'maintenance': {'Magistrate'}, 'custody': {'Family Court'},
            'stridhan': {'Family Court', 'Magistrate'}, 'residence': {'Magistrate'},
            'violence': {'Magistrate'}, 'divorce': {'Family Court'},
            'birth_certificate': {'Registrar'}}.get(issue, set())


def court_for(kept, fired, probs, facts, in_scope):
    # Returns (courts_list, court_source, top2_list).
    # court_source: 'kept' | 'provisional' | 'fallback-escalate' | 'out-of-scope'.
    if kept:
        courts = set()
        if kept & {'hma-24', 'hma-25', 'hma-26', 'hma-27', 'hma-13', 'hma-13b'}:
            courts.add('Family Court')
        if kept & {'gwa-1890', 'hmga-1956-s6', 'welfare-principle'}:
            courts.add('Family Court')
        if kept & {'bnss-144', 'pwdva-17', 'pwdva-18', 'pwdva-19', 'pwdva-20', 'bns-316'}:
            courts.add('Magistrate')
        if 'births-deaths-act-1969' in kept:  # v3 fix: membership, not equality
            courts.add('Registrar')
        return sorted(courts), 'kept', []
    ranked = sorted(probs.items(), key=lambda kv: (kv[1] if kv[1] is not None else -1), reverse=True)
    top2 = [(k, round(v, 4) if v is not None else None) for k, v in ranked[:2]]
    if not in_scope:
        return [], 'out-of-scope', top2
    if len(ranked) >= 2:
        top_prob = ranked[0][1] or 0.0
        second_prob = ranked[1][1] or 0.0
        # Provisional court ONLY on dominance (top >= 0.40 AND margin >= 0.15).
        # Holds for NONE of the 5 empty questions — on this set the fallback
        # below is escalation-only. Do not tune thresholds to force a court.
        if top_prob >= 0.40 and (top_prob - second_prob) >= 0.15:
            return sorted(_court_for_issue(ranked[0][0])), 'provisional', top2
    return ['ESCALATE — no section kept, court cannot be derived'], 'fallback-escalate', top2


# --- v6 FACT DERIVATION FOR NON-16 SETS (STOPGAP — see header note 12) ---
# Frozen text rules. Never tuned per question, never read from GT. A frozen
# annotation (Cell 6 for the 16; the sealed files' five facts) always wins
# over derivation — complete_facts() only fills keys the annotation lacks.
_FILING_RE = re.compile(r'\b(going to file|will file|will be filing|am filing|filing for divorce (next|soon|this)|have (decided|instructed).*fil|going to (file|apply))\b', re.I)
_PENDING_RE = re.compile(r'\b(divorce (case|petition|matter)|case|petition|matter)\b[^.]{0,50}\b(pending|going on|already going on|running in|has been pending|is pending)\b', re.I)
_DECREE_RE = re.compile(r'\bdecree\b.{0,30}\b(passed|already|granted|given)\b|\bdivorce\b.{0,20}\b(decreed|granted)\b', re.I)
_DIVORCE_NEG_RE = re.compile(r"\b(do not|don't|does not|never|not|undecided about)\b.{0,25}\b(want (a )?divorce|divorce)\b", re.I)
_DIVORCE_ASKED_RE = re.compile(r'\b(want|wants|seeking|seek|applying for|file for|filing for|going to file for)\b.{0,25}\b(divorce|end (the |our |this )?marriage|mutual divorce)\b|\bmutual (consent|agreement)\b.{0,25}\b(divorce|end)', re.I)
_DEMAND_TRUE_RE = re.compile(r'\b(asked|demanded|requested)\b.{0,40}\b(return|back|refus)|\brefused to return\b|\bdemand\b.{0,20}\bmade\b', re.I)
_DEMAND_FALSE_RE = re.compile(r"\b(never|not|n't|no)\b.{0,25}\b(asked|demanded)\b|\bhave not asked\b|\bnot asked (for it )?yet\b|\bnever (asked|demanded)\b", re.I)
_VIOLENCE_RE = re.compile(r'\b(beat|beats|beaten|beating|hit|hits|hitting|slap|slapped|threat|threatens|threatened|abuse|abusive|harass|attack|violence|broke my arm|hurt me)\b', re.I)
_RESIDENCE_RE = re.compile(r"\b(thrown out|throw me out|throwing me out|changed the lock|locked (me )?out|cannot enter|can't enter|barred (me )?from (entering|the)|evict|stay in (the )?(house|home|shared (house|home))|return to (our |the )?shared (house|home)|live in (the )?(house|home)|separate place to stay|residence order|right to (reside|stay))\b", re.I)
_MONEY_RE = re.compile(r'\b(maintenance|money|monthly|support|alimony|school fees|expenses|interim maintenance|financial support|run the house)\b', re.I)
_GOLD_RE = re.compile(r'\b(gold|stridhan|streedhan|jewellery|jewelry|ornaments?|bangles?|necklace|earrings?|jewellery box)\b', re.I)


def derive_facts_from_text(q):
    # Returns a full 9-key facts dict derived from the question text only.
    # "Being filed" follows the freeze-note rule exactly: only a definite
    # filing / will-file / going-to-file statement counts. Wanting a divorce,
    # contemplating one, a mutual wish, and conditional "if I file" do NOT.
    return {
        'divorce_asked': bool(_DIVORCE_ASKED_RE.search(q)) and not bool(_DIVORCE_NEG_RE.search(q)),
        'proceeding_pending': bool(_PENDING_RE.search(q)),
        'being_filed': bool(_FILING_RE.search(q)),
        'decree': bool(_DECREE_RE.search(q)),
        # Demand: the FALSE (never/not asked) check runs FIRST — "never asked
        # them to return it" contains "asked ... return", which the TRUE
        # pattern would otherwise catch. A negation before asked/demanded wins.
        'demand_made': (False if _DEMAND_FALSE_RE.search(q) else
                        True if _DEMAND_TRUE_RE.search(q) else None),
        'violence': bool(_VIOLENCE_RE.search(q)),
        'residence_claim': bool(_RESIDENCE_RE.search(q)),
        'money_fact': bool(_MONEY_RE.search(q)),
        'gold_fact': bool(_GOLD_RE.search(q)),
    }


def complete_facts(q, partial=None):
    # Frozen annotations win; derivation fills only the keys they lack.
    # Sealed sets pass their five facts as `partial`; the bank passes None.
    facts = derive_facts_from_text(q)
    if partial:
        for k, v in partial.items():
            if k in facts:
                facts[k] = v
    return facts


# --- Offline shadow test of the HMA 25 gate (no model calls; fails fast) ---
_shadow_sub = {'maintenance:maintenance-root': 'permanent alimony at/after an HMA decree'}
_shadow_facts = {'divorce_asked': False, 'proceeding_pending': False, 'decree': False, 'demand_made': None, 'violence': False}
_k, _d, _e, _g, _cbg = stage2({'maintenance'}, _shadow_sub, _shadow_facts)
assert 'hma-25' not in _k, 'HMA25 gate shadow test FAILED: hma-25 kept with decree=false'
assert _g, 'HMA25 gate shadow test FAILED: no gated_out entry with decree=false'
assert 'hma-25' in _cbg, 'HMA25 gate shadow test FAILED: hma-25 missing from candidates_before_gate'
_k2, _d2, _e2, _g2, _cbg2 = stage2({'maintenance'}, _shadow_sub, {**_shadow_facts, 'decree': True})
assert 'hma-25' in _k2, 'HMA25 gate shadow test FAILED: hma-25 dropped with decree=true'
print('HMA25 gate shadow test: PASS (decree=false -> gated, not kept, in candidates_before_gate; decree=true -> kept)')
# --- Offline shadow test of the D2 HMA 27 gate (no model calls; fails fast) ---
# being_filed=False + proceeding_pending=False -> hma-27 gated out, bns-316 kept.
# Either fact True -> hma-27 kept. Facts dicts here carry being_filed explicitly;
# stage2() also reads it via facts.get('being_filed', False) so older fact dicts
# without the key do not raise KeyError.
_shadow27_sub = {}
_shadow27_base = {'divorce_asked': False, 'proceeding_pending': False, 'being_filed': False, 'decree': False, 'demand_made': True, 'violence': False}
_k27, _d27, _e27, _g27, _cbg27 = stage2({'stridhan'}, _shadow27_sub, _shadow27_base)
assert 'hma-27' not in _k27, 'HMA27 gate shadow test FAILED: hma-27 kept with no proceeding pending/being filed'
assert 'bns-316' in _k27, 'HMA27 gate shadow test FAILED: bns-316 gated with hma-27 (BNS 316 must stay)'
assert any(g.startswith('hma-27: gated (D2)') for g in _g27), 'HMA27 gate shadow test FAILED: no D2 gated_out entry'
assert 'hma-27' in _cbg27, 'HMA27 gate shadow test FAILED: hma-27 missing from candidates_before_gate'
_k27b, _d27b, _e27b, _g27b, _cbg27b = stage2({'stridhan'}, _shadow27_sub, {**_shadow27_base, 'proceeding_pending': True})
assert 'hma-27' in _k27b, 'HMA27 gate shadow test FAILED: hma-27 dropped with proceeding pending'
_k27c, _d27c, _e27c, _g27c, _cbg27c = stage2({'stridhan'}, _shadow27_sub, {**_shadow27_base, 'being_filed': True})
assert 'hma-27' in _k27c, 'HMA27 gate shadow test FAILED: hma-27 dropped with being_filed=True'
print('HMA27 gate shadow test: PASS (no proceeding -> gated, bns-316 kept, D2 reason recorded; pending/being-filed -> kept)')



def run_one_question(t):
    # v6 refactor: the v5 per-question Stage 1 + Stage 2 body, moved
    # verbatim into one reusable function (re-indented only). No threshold,
    # prompt, scoring-logic or fact change. The caller (TEST_SET loop,
    # batch cells, sealed cells) appends the record and prints the
    # one-line summary. t must carry 'id', 'question', 'facts', 'gt_ids',
    # 'gt_labels'. For bank/sealed questions the caller builds 'facts'
    # with complete_facts() above; the 16 use frozen Cell 6 facts.
        q = t['question']
        facts = t['facts']
        t0 = time.perf_counter()
        rec = {'id': t['id'], 'question': q, 'gt_ids': t['gt_ids'],
               'gt_labels': t['gt_labels'], 'facts': facts,
               'stage1_raw': {}, 'stage1_probs': {}, 'stage1_clause_max': {},
               'stage1_final': {}, 'fired_issues': [], 'fired_clause': {},
               'fired_source': {}, 'aux_scores': {}, 'aux_max': {},
               'clauses': [], 'clause_errors': [], 'aux_errors': [],
               'residence_r0': None, 'residence_r1': None,
               'residence_r0_clause_max': None,
               'residence_claim': None, 'residence_gated': None,
               'money_fact': None, 'gold_fact': None,
               'custody_claim': None, 'custody_gated': None,
               'custody_final_pre_gate': None, 'single_issue_guard': None,
               'residence_sub_choice_info': {},
               'divorce_asked_prob': None, 'divorce_pending_prob': None,
               'divorce_signal': 'none',
               'in_scope': None, 'in_scope_prob': None,
               'sub_choices': {}, 'sub_choice_body': {},
               'candidates_before_stage2': [], 'candidates_before_gate': [],
               'gated_out': [], 'kept_ids': [], 'dropped': [],
               'escalations': [], 'escalations_typed': [],
               'court_live': [], 'court_source': None, 'court_top2': [],
               'error': None, 'latency_s': None}
        try:
            gated_out_a = []  # v4: initialised here so the residence fact gate (below) can append before sub-choices run
            # --- full-question pass (baseline, recorded, never deleted) ---
            full_payload = {}
            for name, instr in ISSUES:
                full_payload[f'issue_{name}'] = {'type': 'noul', 'instructions': instr}
            full_payload['issue_residence_r1'] = {'type': 'noul', 'instructions': RESIDENCE_R1}
            full_payload['issue_divorce_pending'] = {'type': 'noul', 'instructions': DIVORCE_PENDING_Q}
            full_payload['issue_in_scope'] = {'type': 'noul', 'instructions': IN_SCOPE_Q}
            ans_full = _ask(full_payload, q)
            raw = {}
            for name, _ in ISSUES:
                raw[name] = _prob_from(ans_full, f'issue_{name}')
            raw['residence_r1'] = _prob_from(ans_full, 'issue_residence_r1')
            raw['divorce_pending'] = _prob_from(ans_full, 'issue_divorce_pending')
            raw['in_scope'] = _prob_from(ans_full, 'issue_in_scope')
            rec['stage1_raw'] = {k: (round(v, 4) if v is not None else None) for k, v in raw.items()}
            rec['stage1_probs'] = dict(rec['stage1_raw'])  # v1 field name, kept for compat
            rec['residence_r0'] = raw.get('residence')  # full-question R0 — the v4 firing prompt
            rec['residence_r1'] = raw.get('residence_r1')
            # --- clause-split passes (one predict call per clause) ---
            clauses = split_clauses(q)
            rec['clauses'] = clauses
            clause_scores = {name: {} for name, _ in ISSUES}
            clause_scores['residence_r1'] = {}
            clause_scores['divorce_pending'] = {}
            clause_scores['in_scope'] = {}
            for clause in clauses:
                payload = {}
                for name, instr in ISSUES:
                    payload[f'issue_{name}'] = {'type': 'noul', 'instructions': instr}
                payload['issue_residence_r1'] = {'type': 'noul', 'instructions': RESIDENCE_R1}
                payload['issue_divorce_pending'] = {'type': 'noul', 'instructions': DIVORCE_PENDING_Q}
                payload['issue_in_scope'] = {'type': 'noul', 'instructions': IN_SCOPE_Q}
                try:
                    ans_c = _ask(payload, clause)
                except Exception as e:
                    rec['clause_errors'].append([clause, repr(e)])
                    continue
                for key in list(clause_scores.keys()):
                    p = _prob_from(ans_c, f'issue_{key}')
                    if p is not None:
                        clause_scores[key][clause] = p
            clause_max = {k: (max(v.values()) if v else None) for k, v in clause_scores.items()}
            rec['stage1_clause_max'] = {k: (round(v, 4) if v is not None else None) for k, v in clause_max.items()}
            rec['residence_r0_clause_max'] = clause_max.get('residence')
            # --- auxiliary layman-phrase prompts (full question) ---
            aux_payload = {}
            for ai, aux in enumerate(AUX_PROMPTS):
                aux_payload[f'aux_{ai}'] = {'type': 'noul', 'instructions': 'Does this question say "' + aux['phrase'] + '" or mean the same thing?'}
            try:
                ans_aux = _ask(aux_payload, q)
            except Exception as e:
                rec['aux_errors'].append(repr(e))
                ans_aux = {}
            aux_scores = {}
            for ai, aux in enumerate(AUX_PROMPTS):
                p = _prob_from(ans_aux, f'aux_{ai}') if f'aux_{ai}' in ans_aux else None
                aux_scores.setdefault(aux['issue'], {})[aux['phrase']] = p
            rec['aux_scores'] = aux_scores
            rec['aux_max'] = {iss: max([v for v in d.values() if v is not None], default=None) for iss, d in aux_scores.items()}
            # --- final scores: max(clause_max, full, aux); residence uses R0 (v4) ---
            TRACK = {'residence': 'residence'}  # v4: R0 selects; R1 never fires
            final = {}
            fired_clause = {}
            fired_source = {}
            for name, _ in ISSUES:
                tk = TRACK.get(name, name)
                best_clause, best_cprob = None, None
                for cl, p in clause_scores.get(tk, {}).items():
                    if best_cprob is None or p > best_cprob:
                        best_clause, best_cprob = cl, p
                aux_m = rec['aux_max'].get(name)
                cands = [(v, s) for v, s in [(best_cprob, 'clause'), (raw.get(tk), 'full'), (aux_m, 'aux')] if v is not None]
                final[name] = max(v for v, s in cands) if cands else None
                fired_clause[name] = best_clause if best_clause is not None else '(full question)'
                fired_source[name] = max(cands, key=lambda vs: vs[0])[1] if cands else None
            rec['stage1_final'] = {k: (round(v, 4) if v is not None else None) for k, v in final.items()}
            rec['fired_clause'] = fired_clause
            rec['fired_source'] = fired_source
            # --- divorce: two signals (asked vs pending) ---
            asked_final = final['divorce']
            pending_final = max([v for v in [clause_max.get('divorce_pending'), raw.get('divorce_pending'), rec['aux_max'].get('divorce_pending')] if v is not None], default=None)
            rec['divorce_asked_prob'] = round(asked_final, 4) if asked_final is not None else None
            rec['divorce_pending_prob'] = round(pending_final, 4) if pending_final is not None else None
            asked_fires = asked_final is not None and asked_final >= THRESHOLDS['divorce_asked']
            pending_fires = pending_final is not None and pending_final >= THRESHOLDS['divorce_pending']
            rec['divorce_signal'] = ('both' if (asked_fires and pending_fires) else
                                      'asked' if asked_fires else
                                      'pending' if pending_fires else 'none')
            rec['stage1_final']['divorce_pending'] = rec['divorce_pending_prob']
            # --- v5 CUSTODY-GATE PATTERNS (Fix 2, verbatim): deterministic
            # per-clause veto patterns. A clause matching the residence
            # co-location pattern can never carry a custody claim. A veto,
            # not a pick — same shape as the v4 residence gate; every pick
            # still comes from agent.predict().
            _CUSTODY_PATTERNS = [re.compile(r"who will get my", re.IGNORECASE), re.compile(r"\b(want|wish|need)\b.{0,30}\bmy (son|daughter|child) to stay with me\b", re.IGNORECASE), re.compile(r"\b(custody|guardianship|guardian)\b", re.IGNORECASE), re.compile(r"\btake (him|her|them) away\b", re.IGNORECASE)]
            _RESIDENCE_COLOC_RE = re.compile(r"\bstay in the (house|home)\b.{0,40}\bwith my (son|daughter|child)\b", re.IGNORECASE)
            def _has_custody_claim(clauses):
                for cl in clauses:
                    if _RESIDENCE_COLOC_RE.search(cl):
                        continue
                    if any(p.search(cl) for p in _CUSTODY_PATTERNS):
                        return True
                return False
            # --- v4 RESIDENCE FACT GATE (Stage 1): residence_claim is a frozen
            # Cell 6 facts annotation (see builder / change log for the rule).
            # If False, residence final is forced to 0.0 and residence can never
            # fire, however high the R0 score is (H2 R0 0.8464, H5 0.4882,
            # H12 0.2177, H10 0.1901, H3 0.1727, H14 0.1550 are all gated out
            # this way). The gate entry is recorded in gated_out (Layer A style).
            # If True, residence fires per the R0 score at threshold 0.15.
            # fired_clause / fired_source for residence stay as computed (they
            # record what WOULD have fired) — only the final score is forced.
            rec['residence_claim'] = bool(facts.get('residence_claim', False))
            if not rec['residence_claim']:
                final['residence'] = 0.0
                rec['stage1_final']['residence'] = 0.0
                gated_out_a.append('residence: gated, no residence claim in facts (residence_claim=false)')
                rec['residence_gated'] = True
            else:
                rec['residence_gated'] = False
            # --- v5 EVIDENCE GATES (Fix 1): money_fact / gold_fact (Cell 6
            # frozen annotations, same style as residence_claim). Veto only:
            # a missing grounding fact forces the final to 0.0; the
            # per-issue threshold still applies on top (maintenance fires
            # only if money_fact True AND final >= 0.10; stridhan only if
            # gold_fact True AND final >= 0.50). H8 rests on the explicit
            # "maintenance"-ask rule; H10's gold fact holds although demand
            # has not yet been made (demand_made stays separate).
            rec['money_fact'] = bool(facts.get('money_fact', False))
            rec['gold_fact'] = bool(facts.get('gold_fact', False))
            if not rec['money_fact']:
                final['maintenance'] = 0.0
                rec['stage1_final']['maintenance'] = 0.0
                gated_out_a.append('evidence gate: maintenance gated, money_fact=false')
            if not rec['gold_fact']:
                final['stridhan'] = 0.0
                rec['stage1_final']['stridhan'] = 0.0
                gated_out_a.append('evidence gate: stridhan gated, gold_fact=false')
            # --- v5 CUSTODY GATE (Fix 2): veto — custody fires only if some
            # clause carries a who-gets-the-child / guardianship claim and
            # is not itself a residence co-location clause ("stay in the
            # house with my daughter"). fired_clause / fired_source stay as
            # computed (they record what WOULD have fired) — only the final
            # score is forced. Pre-gate final recorded for audit.
            # THRESHOLDS['custody'] (0.20), AUX_PROMPTS custody entries,
            # the main custody prompt and split_clauses are untouched.
            rec['custody_final_pre_gate'] = (round(final['custody'], 4)
                                             if final.get('custody') is not None else None)
            rec['custody_claim'] = _has_custody_claim(clauses)
            if not rec['custody_claim']:
                final['custody'] = 0.0
                rec['stage1_final']['custody'] = 0.0
                gated_out_a.append('custody: gated, no who-gets-the-child / guardianship claim in any clause (custody_claim=false)')
                rec['custody_gated'] = True
            else:
                rec['custody_gated'] = False
            # --- v5 SINGLE-ISSUE GUARD (Fix 3): defence-in-depth on top of
            # Fix 1. Arms iff exactly ONE issue clears its OWN per-issue
            # threshold on the FULL-question score alone (raw, BEFORE any
            # veto — dominance must use raw full scores, not vetoed finals).
            # Divorce is judged on its ASKED signal only (>= 0.45).
            # Residence uses the R0 full score, never R1. For any other
            # issue with full <= 0.01, the clause/aux lift must reach 0.80
            # or the issue is blocked (divorce lift also includes the
            # pending clause/full/aux). The residence fact gate runs before
            # the guard and takes precedence (its reason string is the
            # recorded reason where it fires). No in_scope firing gate
            # (rejected: H13 in_scope final 0.0985 is BELOW H7's 0.113).
            # No global aux/clause bar.
            # NOTE: divorce firing reads asked_fires/pending_fires, computed
            # before the guard — a guard veto on divorce zeroes the recorded
            # final only and cannot change the firing decision. On the 16
            # this is inert (H7/H15 divorce asked/pending are both False);
            # flagged for v6, not changed here (spec does not authorise it).
            GUARD_LIFT_BAR = 0.80
            GUARD_FULL_CEILING = 0.01
            _guard_dominant = []
            for _gname, _ in ISSUES:
                if _gname == 'divorce':
                    _gfull, _gthr = raw.get('divorce'), THRESHOLDS['divorce_asked']
                else:
                    _gfull, _gthr = raw.get(_gname), THRESHOLDS[_gname]
                if _gfull is not None and _gfull >= _gthr:
                    _guard_dominant.append((_gname, _gfull))
            _guard_blocked = []
            _guard_armed = len(_guard_dominant) == 1
            if _guard_armed:
                _dom_name, _dom_full = _guard_dominant[0]
                for _gname, _ in ISSUES:
                    if _gname == _dom_name:
                        continue
                    _full_j = raw.get('divorce') if _gname == 'divorce' else raw.get(_gname)
                    if _full_j is None or _full_j > GUARD_FULL_CEILING:
                        continue
                    _lifts = [clause_max.get(_gname), rec['aux_max'].get(_gname)]
                    if _gname == 'divorce':
                        _lifts += [clause_max.get('divorce_pending'),
                                   raw.get('divorce_pending'),
                                   rec['aux_max'].get('divorce_pending')]
                    _lifts = [v for v in _lifts if v is not None]
                    _lift_j = max(_lifts) if _lifts else None
                    if _lift_j is None or _lift_j < GUARD_LIFT_BAR:
                        final[_gname] = 0.0
                        rec['stage1_final'][_gname] = 0.0
                        _lift_disp = _lift_j if _lift_j is not None else 0.0
                        gated_out_a.append(
                            f"single-issue guard: {_gname} blocked (dominant={_dom_name}, "
                            f"full={_full_j:.4f}<=0.01, lift={_lift_disp:.4f}<0.80)")
                        _guard_blocked.append((_gname, round(_full_j, 4),
                                               round(_lift_j, 4) if _lift_j is not None else None))
            rec['single_issue_guard'] = {
                'armed': _guard_armed,
                'dominant': _guard_dominant[0][0] if _guard_armed else None,
                'dominant_full': round(_guard_dominant[0][1], 4) if _guard_armed else None,
                'blocked': _guard_blocked,
            }
            # --- firing (per-issue thresholds; residence = R0 final, fact-gated) ---
            fired = []
            for name, _ in ISSUES:
                if name == 'divorce':
                    if asked_fires or pending_fires:
                        fired.append('divorce')
                    continue
                score, thr = final[name], THRESHOLDS[name]
                if score is not None and score >= thr:
                    fired.append(name)
            rec['fired_issues'] = fired
            # --- in_scope (8th question; clause-max vs full max, threshold 0.50) ---
            in_scope_final = max([v for v in [clause_max.get('in_scope'), raw.get('in_scope')] if v is not None], default=None)
            rec['in_scope_prob'] = round(in_scope_final, 4) if in_scope_final is not None else None
            rec['in_scope'] = bool(in_scope_final is not None and in_scope_final >= THRESHOLDS['in_scope'])
            # --- sub-choices, one call per fired issue, body = its argmax clause ---
            # (gated_out_a was initialised at the top of this try block, v4.)
            sub = {}
            sub_body = {}
            for iss in fired:
                body = fired_clause.get(iss) or q
                if len(body.split()) < 5:
                    body = q
                nodes = []
                for node_id in [ISSUE_NODE.get(iss), ISSUE_EXTRA_NODE.get(iss)]:
                    if node_id and node_id in NODES:
                        node = NODES[node_id]
                        opts = node.get('options')
                        if isinstance(opts, list) and opts and all(isinstance(o, str) for o in opts):
                            opts = list(opts)
                            # HMA 25 gate, Layer A: filter the decree option before offering it.
                            if node_id == 'maintenance:maintenance-root' and not facts['decree']:
                                if 'permanent alimony at/after an HMA decree' in opts:
                                    opts.remove('permanent alimony at/after an HMA decree')
                                    gated_out_a.append("hma-25: option 'permanent alimony at/after an HMA decree' removed from maintenance:maintenance-root before offering (Layer A), decree=false")
                            nodes.append((node_id, opts, node))
                if not nodes:
                    continue
                payload = {}
                for node_id, opts, node in nodes:
                    payload[node_id] = {'type': 'choice', 'instructions': node.get('question', node_id),
                                        'criteria': {f'opt{i}': o for i, o in enumerate(opts)}}
                ans2 = _ask(payload, body)
                for node_id, opts, node in nodes:
                    raw2 = _parse(ans2[node_id]['choice'], 'choice')
                    if isinstance(raw2, str) and raw2.startswith('opt') and raw2[3:].isdigit():
                        raw2 = opts[int(raw2[3:])]
                    sub[node_id] = raw2
                    sub_body[node_id] = body
            rec['sub_choices'] = sub
            rec['sub_choice_body'] = sub_body
            if 'residence' in fired:
                # v4: informational only — the Stage 2 full set never reads this.
                rec['residence_sub_choice_info'] = {k: v for k, v in sub.items() if k.startswith('residence:')}
            # --- Stage 2 + court fallback ---
            kept, dropped, esc_typed, gated_out_b, cbg = stage2(set(fired), sub, facts)
            rec['kept_ids'] = sorted(kept)
            rec['candidates_before_stage2'] = sorted(kept | {d[0] for d in dropped if d[0] in SECTION_IDS})
            rec['candidates_before_gate'] = cbg
            rec['gated_out'] = gated_out_a + gated_out_b
            rec['dropped'] = [[a, b] for a, b in dropped]
            probs_for_court = dict(final)
            probs_for_court['divorce'] = max([v for v in [asked_final, pending_final] if v is not None], default=None)
            courts, court_source, top2 = court_for(kept, set(fired), probs_for_court, facts, rec['in_scope'])
            rec['court_live'] = courts
            rec['court_source'] = court_source
            rec['court_top2'] = [[k, v] for k, v in top2]
            if court_source == 'fallback-escalate':
                esc_typed.append({'kind': 'court', 'text': 'Court: no section kept and question is in scope — escalate to a human; court cannot be derived from sub-threshold scores'})
            rec['escalations_typed'] = esc_typed
            rec['escalations'] = [e['text'] for e in esc_typed]  # plain-text, backward compatible
        except Exception as e:
            rec['error'] = repr(e)
            print(f"{t['id']}: LIVE ERROR — {e!r} (scored as a miss, no fallback)")
        rec['latency_s'] = round(time.perf_counter() - t0, 2)
        return rec

RESULTS = []
for t in TEST_SET:
    _rec = run_one_question(t)
    RESULTS.append(_rec)
    print(f"{_rec['id']}: fired={_rec['fired_issues']} kept={[LABEL_BY_ID.get(s, s) for s in _rec['kept_ids']]} "
          f"court={_rec['court_live']} ({_rec['court_source']}) GT={t['gt_labels']} latency={_rec['latency_s']}s error={_rec['error']}")
print()
print(f'Live run complete: {len(RESULTS)} questions, '
      f"{sum(1 for r in RESULTS if r['error'] is None)}/16 with no model error.")
print('Note: v3 makes up to ~1 (full) + n_clauses + 1 (aux) + n_fired sub-choice calls per question,')
print('so expect a longer run than v1 (188 s) — roughly 10-20 min on CPU.')


In [ ]:
# Cell 8 — v6 score, save, download (16-question REGRESSION vs corrected v2 GT).
# Scoring is the backtest's (exact section-set match, per-section precision /
# recall, forbidden mappings, court correct, escalates correctly) PLUS the
# v6 diagnostics below. Original metrics are kept verbatim so v6 compares
# directly with the v5 live baseline (exact 16/16, mean P/R 1.000/1.000,
# micro P/R 1.000/1.000, TP/FP/FN 57/0/0, court 16/16, escalates 12/16 —
# all computed under the OLD 57-section GT; v6 re-scores the same run shape
# under the corrected 53-section GT, so the numbers are NOT directly
# comparable — D1-A re-bases the story, see build note).
# D1-A + D2: the 16 are scored against the corrected v2 GT (Cell 6). The D2
# gate drops hma-27 where no proceeding is pending/being filed, so K0/H3/H10/H15
# can still read exact. FORBIDDEN_RULES gains the hma-27 rule (HMA 27 with no
# proceeding pending/being filed) — a D2-consequent addition, recorded here.
# DUAL ESCALATES SCORING (expansion brief): every row reports BOTH the old
# boolean escalates score (esc_live from the typed kinds below) AND the
# typed-kinds score (predicted kind set == GT kind set, strict; plus a
# fact-excluded variant). The typed-escalation kinds themselves (Lane 2
# section 4.1) are generated in Cell 7 stage2(). 'fact' is deliberately NOT
# in the scoring set (follow-up questions, not an escalation reason).
# H7/H15 RESOLVED (D3/D4, rater 2026-11-27): escalate_expected kept True for
# both; procedure (H7) and recovery (H15) kinds added. D5 rater confirmation
# of the H2/H4 kinds is still outstanding.
# GATE-FORMULA RATING (gate section 2, frozen) is computed here for the 16,
# with the full arithmetic shown; Fp is NOT MEASURED (Cell 9 prose skipped),
# so the 9.0 cap applies and is printed, not hidden.
# G14 LATENCY: mean/max/total/load aggregates + a per-component PASS/FAIL
# check are printed. Expect FAIL on CPU (v5 mean was 61.9 s); T4 is
# unverified — no T4 number is claimed anywhere in this notebook.
# FORBIDDEN METRICS: forbidden_before_recheck / forbidden_after_recheck are
# computed exactly as in v1 (from candidates_before_stage2). NEW:
# forbidden_proposed_before_gate (from candidates_before_gate) and
# gated_out_total (+ gated_out_layerB_total). After the HMA 25 Layer A gate
# the option cannot be picked, so forbidden_proposed_before_gate is
# 0-by-construction for the HMA 25 route — the honest gate check is the
# Cell 7 shadow test (printed PASS) + H5/H9 kept_ids byte-identical to v1.
# Prose-level forbidden errors remain NOT MEASURED (no answer prose generated).

import json

FORBIDDEN_RULES = [
    ('hma-13', 'HMA 13/13B with no divorce asked', lambda f: not f['divorce_asked']),
    ('hma-13b', 'HMA 13/13B with no divorce asked', lambda f: not f['divorce_asked']),
    ('hma-24', 'HMA 24 with no proceeding pending', lambda f: not f['proceeding_pending']),
    ('hma-25', 'HMA 25 with no decree', lambda f: not f['decree']),
    ('hma-26', 'HMA 26 with no proceeding pending', lambda f: not f['proceeding_pending']),
    ('hma-27', 'HMA 27 with no proceeding pending or being filed', lambda f: not (f['proceeding_pending'] or f.get('being_filed', False))),
]

# Section id -> courts it supports (same mapping as court_for in Cell 7).
SECTION_COURT = {}
for _s in ('hma-24', 'hma-25', 'hma-26', 'hma-27', 'hma-13', 'hma-13b', 'hma-9', 'gwa-1890', 'hmga-1956-s6', 'welfare-principle'):
    SECTION_COURT.setdefault(_s, set()).add('Family Court')
for _s in ('bnss-144', 'pwdva-17', 'pwdva-18', 'pwdva-19', 'pwdva-20', 'pwdva-22', 'bns-316'):
    SECTION_COURT.setdefault(_s, set()).add('Magistrate')
SECTION_COURT.setdefault('births-deaths-act-1969', set()).add('Registrar')

EMPTY5 = ['H1', 'H4', 'H8', 'H11', 'H14']


def _pr(pred, gt):
    pred, gt = set(pred), set(gt)
    tp = len(pred & gt)
    prec = tp / len(pred) if pred else (1.0 if not gt else 0.0)
    rec = tp / len(gt) if gt else (1.0 if not pred else 0.0)
    return tp, len(pred) - tp, len(gt) - tp, prec, rec

# --- v6 shared scoring helpers (used by Cell 8 itself, every batch cell,
# the sealed cells and the final cell). Pure scoring — no model calls, no
# threshold / fact / gate change. ---
SCORING_KINDS = {'quantum', 'court', 'demand', 'custody', 'residence', 'recovery', 'procedure', 'safety'}
# 'fact' is deliberately NOT in SCORING_KINDS (follow-up questions, not a reason).

COURT_NORMALIZE = {'Judicial Magistrate First Class': 'Magistrate',
                   'Registrar of Births and Deaths': 'Registrar',
                   'Family Court': 'Family Court', 'Magistrate': 'Magistrate',
                   'Registrar': 'Registrar'}

# Sealed expected_escalations types -> v6 kind vocabulary. Types with no v6
# equivalent (human-lawyer handoff, enforcement, ask-decision) map to None
# and are reported per question as unmapped — never silently forced into a kind.
SEALED_KIND_MAP = {'quantum': 'quantum', 'safety': 'safety',
                   'written-demand-first': 'demand', 'registrar-procedure': 'procedure',
                   'ask_missing_fact': 'fact', 'proof': 'fact',
                   'human-lawyer': None, 'enforcement': None, 'ask-decision': None}


def _wilson(k, n, z=1.96):
    # 95% Wilson interval for a proportion k/n (plan section 3.3 reading guide).
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    m = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5)
    return (round(max(0.0, (c - m) / d), 4), round(min(1.0, (c + m) / d), 4))


def bank_pairs(records):
    # Build score_batch pairs for brutal-bank records (BANK_BY_ID, Cell 5b).
    pairs = []
    for r in records:
        q = BANK_BY_ID[r['id']]
        pairs.append({'id': r['id'], 'category': q['category'],
                      'gt_ids': list(q['gt_sections']),
                      'gt_court_expected': [] if q['gt_court'] == 'none' else [q['gt_court']],
                      'gt_escalates': bool(q['gt_escalates']),
                      'gt_kinds': set(q.get('gt_escalation_kinds', [])), 'rec': r})
    return pairs


def sealed_pairs(records, sealed_json):
    # Build score_batch pairs for a sealed set. Court names normalized
    # (JMFC -> Magistrate, Registrar of Births and Deaths -> Registrar).
    # Section ids need NO renaming: the v5 live JSON shows the pipeline
    # already emits hmga-1956-s6 / welfare-principle exactly as sealed GT
    # does (identity mapping, verified — see build note). Sealed gt_labels
    # are issue words ('maintenance', ...) and are never used for scoring.
    by_id = {q['id']: q for q in sealed_json['questions']}
    pairs = []
    for r in records:
        q = by_id[r['id']]
        mapped = set()
        unmapped = []
        for e in q.get('expected_escalations', []):
            m = SEALED_KIND_MAP.get(e.get('type'))
            if m is None:
                unmapped.append(e.get('type'))
            else:
                mapped.add(m)
        pairs.append({'id': r['id'], 'category': q.get('primary_issue', ''),
                      'gt_ids': list(q['gt_sections']),
                      'gt_court_expected': [COURT_NORMALIZE[c] for c in q.get('expected_court', [])],
                      'gt_escalates': bool(q['escalates']), 'gt_kinds': mapped,
                      'gt_kinds_unmapped': unmapped, 'rec': r,
                      'f20': r['id'] == 'F20'})
    return pairs


def score_batch(pairs, label):
    # Scores a list of pairs (bank or sealed) with the frozen definitions:
    # exact = set equality (H12 0/0 convention via _pr); court = lead-forum
    # containment (expected lead forum(s) in court_live; empty GT expects
    # empty court); escalates boolean = any SCORING_KINDS kind present;
    # escalates kinds = predicted kind set == GT kind set (strict) and the
    # fact-excluded variant. F20 RULE (freeze note section 11): F20 is
    # out-of-scope with gt_escalates=True as a human-lawyer handoff; it is
    # scored correct iff the system declined (empty kept, empty court).
    # Returns (rows, agg) and prints the batch summary.
    rows = []
    tot_tp = tot_fp = tot_fn = 0
    for p in pairs:
        r = p['rec']
        pred, gt = r['kept_ids'], p['gt_ids']
        tp, fp, fn, prec, rec = _pr(pred, gt)
        tot_tp += tp; tot_fp += fp; tot_fn += fn
        facts = r.get('facts', {})
        fb = [rule for s, rule, cond in FORBIDDEN_RULES
              if s in r.get('candidates_before_stage2', []) and cond(facts)]
        fa = [rule for s, rule, cond in FORBIDDEN_RULES
              if s in pred and cond(facts)]
        fbg = [rule for s, rule, cond in FORBIDDEN_RULES
               if s in r.get('candidates_before_gate', []) and cond(facts)]
        exp_court = p['gt_court_expected']
        if not exp_court:
            court_ok = (r['court_live'] == [])
        else:
            court_ok = set(exp_court) <= set(r['court_live'])
        typed = r.get('escalations_typed', []) or []
        kinds = {e.get('kind') for e in typed if isinstance(e, dict)}
        pred_kinds = kinds & SCORING_KINDS
        esc_live = bool(pred_kinds)
        gt_kinds = set(p['gt_kinds'])
        if p.get('f20'):
            esc_ok = (pred == [] and r['court_live'] == [])
        else:
            esc_ok = esc_live == p['gt_escalates']
        kinds_ok = pred_kinds == gt_kinds
        kinds_ok_fe = pred_kinds == (gt_kinds - {'fact'})
        silent = bool(gt) and not pred
        rows.append({'id': p['id'], 'category': p.get('category', ''),
                     'exact': set(pred) == set(gt),
                     'precision': round(prec, 3), 'recall': round(rec, 3),
                     'tp': tp, 'fp': fp, 'fn': fn,
                     'pred_ids': sorted(pred), 'gt_ids': sorted(gt),
                     'forbidden_before_recheck': fb, 'forbidden_after_recheck': fa,
                     'forbidden_proposed_before_gate': fbg,
                     'gated_out': r.get('gated_out', []),
                     'court_live': r['court_live'], 'court_expected': exp_court,
                     'court_correct': court_ok,
                     'escalates_live': esc_live, 'escalates_expected': p['gt_escalates'],
                     'escalates_correct': esc_ok,
                     'escalation_kinds': sorted(pred_kinds),
                     'gt_escalation_kinds': sorted(gt_kinds),
                     'escalates_kinds_correct': kinds_ok,
                     'escalates_kinds_correct_fact_excluded': kinds_ok_fe,
                     'gt_kinds_unmapped': p.get('gt_kinds_unmapped', []),
                     'f20_rule_applied': bool(p.get('f20')),
                     'silent': silent,
                     'error': r['error'], 'latency_s': r['latency_s']})
    n = len(rows)
    n_exact = sum(1 for x in rows if x['exact'])
    micro_p = tot_tp / (tot_tp + tot_fp) if (tot_tp + tot_fp) else 0.0
    micro_r = tot_tp / (tot_tp + tot_fn) if (tot_tp + tot_fn) else 0.0
    micro_f1 = (2 * micro_p * micro_r / (micro_p + micro_r)) if (micro_p + micro_r) else 0.0
    per_cat = {}
    for x in rows:
        c = x['category'] or '(no category)'
        a, b = per_cat.get(c, (0, 0))
        per_cat[c] = (a + (1 if x['exact'] else 0), b + 1)
    per_cat_exact = {c: f'{a}/{b}' for c, (a, b) in sorted(per_cat.items())}
    per_cat_min = min([a / b for a, b in per_cat.values()]) if per_cat else 0.0
    lats = [x['latency_s'] for x in rows if x['latency_s'] is not None]
    agg = {'label': label, 'n': n, 'exact': f'{n_exact}/{n}', 'exact_count': n_exact,
           'mean_precision': round(sum(x['precision'] for x in rows) / n, 3) if n else 0.0,
           'mean_recall': round(sum(x['recall'] for x in rows) / n, 3) if n else 0.0,
           'micro_precision': round(micro_p, 3), 'micro_recall': round(micro_r, 3),
           'micro_f1': round(micro_f1, 3), 'micro_tp_fp_fn': [tot_tp, tot_fp, tot_fn],
           'court_correct': f"{sum(1 for x in rows if x['court_correct'])}/{n}",
           'escalates_bool_correct': f"{sum(1 for x in rows if x['escalates_correct'])}/{n}",
           'escalates_kinds_correct_strict': f"{sum(1 for x in rows if x['escalates_kinds_correct'])}/{n}",
           'escalates_kinds_correct_fact_excluded': f"{sum(1 for x in rows if x['escalates_kinds_correct_fact_excluded'])}/{n}",
           'silent_count': sum(1 for x in rows if x['silent']),
           'silent_ids': [x['id'] for x in rows if x['silent']],
           'forbidden_before_recheck_total': sum(len(x['forbidden_before_recheck']) for x in rows),
           'forbidden_ids': [x['id'] for x in rows if x['forbidden_before_recheck']],
           'model_errors': sum(1 for x in rows if x['error']),
           'model_error_ids': [x['id'] for x in rows if x['error']],
           'per_category_exact': per_cat_exact,
           'per_category_min_exact': round(per_cat_min, 3),
           'escalates_on_empty_kept': sum(1 for x in rows if not x['pred_ids'] and x['escalation_kinds']),
           'hma27_gt_note_ids': [x['id'] for x in rows if 'hma-27' in x['gt_ids'] and 'hma-27' not in x['pred_ids']],
           'latency_mean_s': round(sum(lats) / len(lats), 2) if lats else None,
           'latency_max_s': round(max(lats), 2) if lats else None}
    print(f"--- {label}: exact {agg['exact']} | micro P/R/F1 {agg['micro_precision']}/{agg['micro_recall']}/{agg['micro_f1']} "
          f"TP/FP/FN {tot_tp}/{tot_fp}/{tot_fn} | court {agg['court_correct']} | "
          f"esc-bool {agg['escalates_bool_correct']} | esc-kinds strict {agg['escalates_kinds_correct_strict']} "
          f"(fact-excl {agg['escalates_kinds_correct_fact_excluded']}) | silent {agg['silent_count']} | "
          f"Fb {agg['forbidden_before_recheck_total']} | errors {agg['model_errors']} | "
          f"per-cat min {agg['per_category_min_exact']} | latency mean {agg['latency_mean_s']}s ---")
    return rows, agg



rows = []
tot_tp = tot_fp = tot_fn = 0
for t, r in zip(TEST_SET, RESULTS):
    pred, gt = r['kept_ids'], t['gt_ids']
    tp, fp, fn, prec, rec = _pr(pred, gt)
    tot_tp += tp; tot_fp += fp; tot_fn += fn
    fb = [rule for s, rule, cond in FORBIDDEN_RULES
          if s in r['candidates_before_stage2'] and cond(t['facts'])]
    fa = [rule for s, rule, cond in FORBIDDEN_RULES
          if s in pred and cond(t['facts'])]
    fbg = [rule for s, rule, cond in FORBIDDEN_RULES
           if s in r.get('candidates_before_gate', []) and cond(t['facts'])]
    court_ok = set(r['court_live']) == set(t['court_expected'])
    kept_gt = set(pred) & set(gt)
    if not t['court_expected']:
        cfk = court_ok
    else:
        cfk = court_ok and all(
            any(c in SECTION_COURT.get(s, set()) for s in kept_gt)
            for c in t['court_expected'])
    typed = r.get('escalations_typed', []) or []
    kinds = {e.get('kind') for e in typed if isinstance(e, dict)}
    esc_live = bool(kinds & {'quantum', 'court', 'demand', 'custody', 'residence', 'recovery', 'procedure', 'safety'})  # v6: Lane 2 kinds + safety (Cell 7); 'fact' deliberately excluded (follow-up questions, not a reason)
    esc_live_any = bool(kinds)  # diagnostic: any typed kind incl. demand
    esc_live_bool = bool(r.get('escalations'))  # diagnostic: v1 rule
    esc_ok = esc_live == t['escalate_expected']
    gt_kinds = set(t.get('gt_escalation_kinds', []))
    pred_kinds = kinds & {'quantum', 'court', 'demand', 'custody', 'residence', 'recovery', 'procedure', 'safety'}
    kinds_ok = pred_kinds == gt_kinds
    kinds_ok_fe = pred_kinds == (gt_kinds - {'fact'})
    rows.append({'id': t['id'], 'exact': set(pred) == set(gt),
                 'precision': round(prec, 3), 'recall': round(rec, 3),
                 'tp': tp, 'fp': fp, 'fn': fn,
                 'pred_labels': [LABEL_BY_ID.get(s, s) for s in pred],
                 'gt_labels': t['gt_labels'],
                 'forbidden_before_recheck': fb, 'forbidden_after_recheck': fa,
                 'forbidden_proposed_before_gate': fbg,
                 'gated_out': r.get('gated_out', []),
                 'court_live': r['court_live'], 'court_expected': t['court_expected'],
                 'court_correct': court_ok,
                 'court_from_kept_ok': cfk,
                 'court_source': r.get('court_source'),
                 'escalates_live': esc_live, 'escalates_expected': t['escalate_expected'],
                 'escalates_correct': esc_ok,
                 'escalates_live_any_typed': esc_live_any,
                 'escalates_live_bool_plain': esc_live_bool,
                 'escalation_kinds': sorted(pred_kinds),
                 'gt_escalation_kinds': sorted(gt_kinds),
                 'escalates_kinds_correct': kinds_ok,
                 'escalates_kinds_correct_fact_excluded': kinds_ok_fe,
                 'fired_issues': r.get('fired_issues', []),
                 'fired_clause': r.get('fired_clause', {}),
                 'error': r['error'], 'latency_s': r['latency_s']})

rows_by_id = {x['id']: x for x in rows}
rec_by_id = {r['id']: r for r in RESULTS}
n = len(rows)
n_exact = sum(1 for x in rows if x['exact'])
micro_p = tot_tp / (tot_tp + tot_fp) if (tot_tp + tot_fp) else 0.0
micro_r = tot_tp / (tot_tp + tot_fn) if (tot_tp + tot_fn) else 0.0

# --- FP by issue (v5: which issues still produce FPs; expect all zero) ---
# Section -> issue map. PWDVA 18 is shared (residence full set /
# violence fallback) and is counted as 'residence-or-violence'.
SECTION_ISSUE = {
    'bnss-144': 'maintenance', 'pwdva-20': 'maintenance',
    'hma-24': 'maintenance', 'hma-25': 'maintenance',
    'gwa-1890': 'custody', 'hmga-1956-s6': 'custody',
    'welfare-principle': 'custody', 'hma-26': 'custody',
    'hma-27': 'stridhan', 'bns-316': 'stridhan',
    'pwdva-17': 'residence', 'pwdva-19': 'residence',
    'pwdva-18': 'residence-or-violence',
    'hma-13': 'divorce', 'hma-13b': 'divorce', 'hma-9': 'divorce',
    'pwdva-22': 'violence',
    'births-deaths-act-1969': 'birth_certificate',
}
fp_by_section = {}
fp_by_issue = {}
for _x in rows:
    _pred = set(rec_by_id[_x['id']]['kept_ids'])
    _gt = set(rec_by_id[_x['id']]['gt_ids'])
    for _s in sorted(_pred - _gt):
        fp_by_section[_s] = fp_by_section.get(_s, 0) + 1
        _iss = SECTION_ISSUE.get(_s, 'unknown')
        fp_by_issue[_iss] = fp_by_issue.get(_iss, 0) + 1

# --- empty-5 slice ---
e5 = [x for x in rows if x['id'] in EMPTY5]
e5_exact = sum(1 for x in e5 if x['exact'])
# --- H2 / H12 checks ---
h2_rec = rec_by_id.get('H2', {})
h2_res_fired = 'residence' in h2_rec.get('fired_issues', [])
h2_pwdva_kept = sorted(set(h2_rec.get('kept_ids', [])) & {'pwdva-17', 'pwdva-18', 'pwdva-19'})
h12_rec = rec_by_id.get('H12', {})
h12_row = rows_by_id.get('H12', {})
h12_empty = (h12_rec.get('kept_ids', None) == [] and h12_rec.get('court_live', None) == []
             and h12_row.get('escalates_live') is False)
# --- residence R0 vs R1 comparison (v4: CORRECTED key lookup) ---
# v3 read stage1_clause_max['residence_r0'] (a key that does not exist ->
# always None) for R0 clause-max and ['residence'] (the R0 clause-max) for
# R1 clause-max, so the v3 table mislabelled R0 clause-max as R1. v4 reads
# the true keys: R0 clause-max = stage1_clause_max['residence'],
# R1 clause-max = stage1_clause_max['residence_r1']. R0 final here =
# max(R0 full, R0 clause-max, aux residence max) — the same max Cell 7
# fires on. R1 final is the same max with R1 in place of R0 (diagnostic
# only; R1 never fires in v4). residence_gated comes from Cell 7.
res_rows = []
for t, r in zip(TEST_SET, RESULTS):
    r0_full = r.get('residence_r0')
    r1_full = r.get('residence_r1')
    r0_cl = (r.get('stage1_clause_max') or {}).get('residence')
    r1_cl = (r.get('stage1_clause_max') or {}).get('residence_r1')
    aux_res = (r.get('aux_max') or {}).get('residence')
    r0_final = max([v for v in (r0_full, r0_cl, aux_res) if v is not None], default=None)
    r1_final = max([v for v in (r1_full, r1_cl, aux_res) if v is not None], default=None)
    fired_r0 = r0_final is not None and r0_final >= 0.15
    fired_r1 = r1_final is not None and r1_final >= 0.15
    res_rows.append({'id': t['id'], 'r0_full': r0_full, 'r1_full': r1_full,
                     'r0_clause_max': r0_cl, 'r1_clause_max': r1_cl,
                     'aux_residence_max': aux_res, 'r0_final': r0_final,
                     'r1_final': r1_final,
                     'residence_claim': r.get('residence_claim'),
                     'residence_gated': r.get('residence_gated'),
                     'fired_r0_at_0.15': fired_r0, 'fired_r1': fired_r1,
                     'residence_fired': 'residence' in r.get('fired_issues', [])})
gated_total = sum(len(x['gated_out']) for x in rows)
gated_layerB_total = sum(1 for x in rows for g in x['gated_out'] if 'Layer B' in str(g))

# --- v6 gate-formula rating (gate section 2, frozen) for the 16 ---
# E = exact share, Rm = mean recall, Pm = mean precision, C = court share,
# S = escalates-boolean share, Fb = forbidden-before-recheck count.
# Fp is NOT MEASURED (Cell 9 prose skipped) so the Fp term is 0 by
# non-measurement — NOT by a 0 count — and the 9.0 cap applies.
_fb_total = sum(len(x['forbidden_before_recheck']) for x in rows)
_E = n_exact / n
_Rm = sum(x['recall'] for x in rows) / n
_Pm = sum(x['precision'] for x in rows) / n
_C = sum(1 for x in rows if x['court_correct']) / n
_S = sum(1 for x in rows if x['escalates_correct']) / n
_rating_raw = 10 * (0.60 * _E + 0.15 * _Rm + 0.10 * _Pm + 0.075 * _C + 0.075 * _S) - 0.5 * _fb_total
_rating_raw = max(0.0, min(10.0, round(_rating_raw, 1)))
_rating_capped = min(_rating_raw, 9.0)  # prose NOT MEASURED -> cap 9.0 (also Fb>=1 -> <=9.0)
_lats = [x['latency_s'] for x in rows if x['latency_s'] is not None]
_lat_mean = round(sum(_lats) / len(_lats), 2) if _lats else None
_lat_max = round(max(_lats), 2) if _lats else None
_lat_total = round(sum(_lats), 1) if _lats else None
_load_s = (LOAD_INFO or {}).get('load_seconds')
_g14 = {'mean_le_15': (_lat_mean is not None and _lat_mean <= 15),
        'max_le_30': (_lat_max is not None and _lat_max <= 30),
        'run_le_240': (_lat_total is not None and _lat_total <= 240),
        'load_le_60': (_load_s is not None and _load_s <= 60)}
_g14['pass'] = all(_g14.values())

agg = {
    'notebook_version': 'v6-brutal (D1-A corrected GT 53 sections + D2 gate + Lane 2 typed kinds; v3 graph 34/48/79; v3 thresholds M 0.10 / C 0.20 / R 0.15 / D-asked 0.45 / D-pend 0.12)',
    'system': 'Laya 421M (classic ModernBERT checkpoint) + v2 two-stage graph, LIVE in Colab',
    'loader': LOAD_INFO,
    'n_questions': n,
    'exact_section_set_match': f'{n_exact}/{n}',
    'mean_precision': round(sum(x['precision'] for x in rows) / n, 3),
    'mean_recall': round(sum(x['recall'] for x in rows) / n, 3),
    'micro_precision': round(micro_p, 3),
    'micro_recall': round(micro_r, 3),
    'micro_tp_fp_fn': [tot_tp, tot_fp, tot_fn],
    'fp_total': tot_fp,
    'fp_by_section': fp_by_section,
    'fp_by_issue': fp_by_issue,
    'empty5_exact': f'{e5_exact}/5',
    'empty5_court_correct': f"{sum(1 for x in e5 if x['court_correct'])}/5",
    'empty5_escalates_correct': f"{sum(1 for x in e5 if x['escalates_correct'])}/5",
    'forbidden_before_recheck_total': sum(len(x['forbidden_before_recheck']) for x in rows),
    'forbidden_after_recheck_total': sum(len(x['forbidden_after_recheck']) for x in rows),
    'forbidden_proposed_before_gate_total': sum(len(x['forbidden_proposed_before_gate']) for x in rows),
    'gated_out_total': gated_total,
    'gated_out_layerB_total': gated_layerB_total,
    'court_correct': f"{sum(1 for x in rows if x['court_correct'])}/{n}",
    'court_from_kept_correct': f"{sum(1 for x in rows if x['court_from_kept_ok'])}/{n}",
    'escalates_correct': f"{sum(1 for x in rows if x['escalates_correct'])}/{n}",
    'escalates_correct_any_typed_diagnostic': f"{sum(1 for x in rows if x['escalates_live_any_typed'] == x['escalates_expected'])}/{n}",
    'escalates_correct_bool_plain_diagnostic': f"{sum(1 for x in rows if x['escalates_live_bool_plain'] == x['escalates_expected'])}/{n}",
    'escalates_kinds_correct': f"{sum(1 for x in rows if x['escalates_kinds_correct'])}/{n}",
    'escalates_kinds_correct_fact_excluded': f"{sum(1 for x in rows if x['escalates_kinds_correct_fact_excluded'])}/{n}",
    'escalates_on_empty_kept': sum(1 for x in rows if not x['pred_labels'] and x['escalation_kinds']),
    'gate_formula_rating_raw': _rating_raw,
    'gate_formula_rating_capped_9.0': _rating_capped,
    'gate_formula_inputs': {'E': round(_E, 3), 'Rm': round(_Rm, 3), 'Pm': round(_Pm, 3), 'C': round(_C, 3), 'S': round(_S, 3), 'Fb': _fb_total, 'Fp': 'NOT MEASURED'},
    'latency_mean_s': _lat_mean, 'latency_max_s': _lat_max, 'latency_total_s': _lat_total,
    'latency_load_s': _load_s, 'g14_check': _g14,
    'questions_with_model_error': sum(1 for x in rows if x['error']),
    'h2_residence_fired': h2_res_fired,
    'h2_pwdva_17_18_19_kept': h2_pwdva_kept,
    'h12_stays_empty': h12_empty,
    'residence_fired_r0_count_at_0.15': sum(1 for x in res_rows if x['fired_r0_at_0.15']),
    'residence_fired_r1_count_at_0.15_diagnostic': sum(1 for x in res_rows if x['fired_r1']),
    'residence_fired_count': sum(1 for x in res_rows if x['residence_fired']),
    'residence_gated_count': sum(1 for x in res_rows if x['residence_gated']),
    'prose_forbidden_mappings': 'NOT MEASURED — Cell 9 prose renderer SKIPPED in v6 (see build note); no answer prose generated, Fp not measured, 9.0 cap applies',
    'h7_h15_escalate_resolution': 'RESOLVED (D3/D4, rater 2026-11-27): H7/H15 escalate_expected kept True; procedure (H7) and recovery (H15) kinds added (Lane 2). D5 rater confirmation of H2/H4 kinds still outstanding.',
    'walk_ceiling_reference': 'Graph-walk ceiling was 16/16 exact (deterministic walk, NOT live). Do not compare live numbers to it as if it were a baseline.',
    'v1_baseline_reference': 'v1 live: exact 5/16, mean P/R 0.656/0.492, micro P/R 0.875/0.368, TP/FP/FN 21/3/36, court 10/16, escalates 11/16, forbidden-before-recheck 2.',
    'v3_baseline_reference': 'v3 live: exact 5/16, mean P/R 0.673/0.866, micro P/R 0.671/0.895, TP/FP/FN 51/25/6, court 11/16, escalates 14/16, residence fired R1 11/16 vs R0 5/16.',
    'v4r_baseline_reference': 'v4r live: exact 10/16, mean P/R 0.833/1.000, micro P/R 0.826/1.000, TP/FP/FN 57/12/0, court 13/16, escalates 14/16, residence fired 4/16, gated 12/16.',
}
print('=== LIVE Laya + v2 graph — Colab results (v6-brutal regression) ===')
for k, v in agg.items():
    print(f'{k}: {v}')
print()
print(f"{'ID':4s} {'exact':>5s} {'prec':>5s} {'rec':>5s} {'fb':>2s} {'fbg':>3s} {'court':>5s} {'esc':>3s}  predicted sections")
for x in rows:
    print(f"{x['id']:4s} {str(x['exact']):>5s} {x['precision']:5.3f} {x['recall']:5.3f} "
          f"{len(x['forbidden_before_recheck']):2d} {len(x['forbidden_proposed_before_gate']):3d} {str(x['court_correct']):>5s} {str(x['escalates_correct']):>3s}  "
          f"{', '.join(x['pred_labels']) if x['pred_labels'] else '(none)'}")
print()
print('fb = forbidden BEFORE Stage 2 recheck (v1 metric). fbg = forbidden PROPOSED before the HMA25 gate.')
print('court = court correct, esc = escalates correctly (boolean, v6 kind set). Kinds strict/fact-excluded are in the aggregate above.')
print()
print('=== Empty-5 detail (H1/H4/H8/H11/H14) ===')
print(f"{'ID':4s} {'exact':>5s} {'prec':>5s} {'rec':>5s} {'TP':>2s} {'FP':>2s} {'FN':>2s} {'court':>5s} {'esc':>3s}  fired issues / fired clauses")
for x in e5:
    fc = x['fired_clause'] or {}
    fc_s = '; '.join(f"{k}={str(v)[:55]}" for k, v in fc.items() if k in x['fired_issues'])
    print(f"{x['id']:4s} {str(x['exact']):>5s} {x['precision']:5.3f} {x['recall']:5.3f} "
          f"{x['tp']:2d} {x['fp']:2d} {x['fn']:2d} {str(x['court_correct']):>5s} {str(x['escalates_correct']):>3s}  "
          f"{', '.join(x['fired_issues']) if x['fired_issues'] else '(none)'} | {fc_s if fc_s else '(no fired clause)'}")
print()
print('=== Residence R0 vs R1 (v4: R0 fires, fact-gated; R1 diagnostic only) ===')
print(f"{'ID':4s} {'R0 full':>7s} {'R1 full':>7s} {'R0 clmax':>8s} {'R1 clmax':>8s} {'aux':>7s} {'R0 fin':>7s} {'claim':>5s} {'gated':>5s} {'fired':>5s}")
for x in res_rows:
    def _f(v):
        return f'{v:.4f}' if v is not None else '  None'
    print(f"{x['id']:4s} {_f(x['r0_full']):>7s} {_f(x['r1_full']):>7s} {_f(x['r0_clause_max']):>8s} {_f(x['r1_clause_max']):>8s} {_f(x['aux_residence_max']):>7s} {_f(x['r0_final']):>7s} "
          f"{str(x['residence_claim']):>5s} {str(x['residence_gated']):>5s} {str(x['residence_fired']):>5s}")
print()
print(f"H2 check: residence fired={h2_res_fired} (v1: True at 0.8247), PWDVA 17/18/19 kept={h2_pwdva_kept} (v1: all three — target: none).")
print(f"H12 check: stays empty={h12_empty} (kept=[], court=[], escalates=False). Must stay True.")
print(f"Gated out total={gated_total} (Layer B picks blocked={gated_layerB_total}); forbidden proposed before gate={agg['forbidden_proposed_before_gate_total']} (v1 forbidden-before-recheck was 2).")
print()
print(f"Escalates (dual scoring): boolean {agg['escalates_correct']} | typed kinds strict {agg['escalates_kinds_correct']} | kinds fact-excluded {agg['escalates_kinds_correct_fact_excluded']} | kinds on empty kept={agg['escalates_on_empty_kept']} (must be 0).")
print(f"Gate-formula rating (16): raw {agg['gate_formula_rating_raw']} -> capped {agg['gate_formula_rating_capped_9.0']} (prose NOT MEASURED, cap 9.0 applies; Fp term is 0 by non-measurement, NOT by a 0 count). Inputs: {agg['gate_formula_inputs']}.")
print(f"G14 latency: mean {agg['latency_mean_s']}s (<=15: {_g14['mean_le_15']}) | max {agg['latency_max_s']}s (<=30: {_g14['max_le_30']}) | run total {agg['latency_total_s']}s (<=240: {_g14['run_le_240']}) | load {agg['latency_load_s']}s (<=60: {_g14['load_le_60']}) | G14 overall: {_g14['pass']} — expect FAIL on CPU (v5 mean was 61.9s); T4 unverified, no T4 number claimed.")

out = {'aggregate': agg, 'per_question': rows, 'residence_r0_vs_r1': res_rows,
       'empty5': e5, 'raw_live_runs': RESULTS, 'load_info': LOAD_INFO}
with open('live-laya-colab-results-v6-brutal-regression.json', 'w') as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
md = ['# Live Laya inference test — Colab run v6-brutal regression (Laya 421M + v3 graph)',
      '',
      '## Aggregate (live, measured in Colab)',
      '']
for k, v in agg.items():
    md.append(f'- **{k}**: {v}')
md += ['', '## Per-question', '',
       '| ID | Exact | Precision | Recall | Forbidden (before recheck) | Forbidden (proposed before gate) | Court | Escalates | Predicted |',
       '|----|-------|-----------|--------|----------------------------|----------------------------------|-------|-----------|-----------|']
for x in rows:
    md.append(f"| {x['id']} | {x['exact']} | {x['precision']} | {x['recall']} | "
              f"{len(x['forbidden_before_recheck'])} | {len(x['forbidden_proposed_before_gate'])} | {x['court_correct']} | {x['escalates_correct']} | "
              f"{', '.join(x['pred_labels']) if x['pred_labels'] else '(none)'} |")
md += ['', '## Empty-5 detail (H1/H4/H8/H11/H14)', '',
       '| ID | Exact | Precision | Recall | TP | FP | FN | Court | Escalates | Fired issues |',
       '|----|-------|-----------|--------|----|----|----|-------|-----------|--------------|']
for x in e5:
    md.append(f"| {x['id']} | {x['exact']} | {x['precision']} | {x['recall']} | {x['tp']} | {x['fp']} | {x['fn']} | "
              f"{x['court_correct']} | {x['escalates_correct']} | {', '.join(x['fired_issues']) if x['fired_issues'] else '(none)'} |")
md += ['', '## Residence R0 vs R1 (v4: R0 fires, fact-gated; R1 diagnostic only)', '',
       '| ID | R0 full | R1 full | R0 clause-max | R1 clause-max | Aux residence max | R0 final | Residence claim | Gated | Residence fired |',
       '|----|---------|---------|---------------|---------------|-------------------|----------|-----------------|-------|---------------|']
md += ['', '## FP by issue (v5: expect all zero)', '']
if fp_by_issue:
    for _iss in sorted(fp_by_issue):
        md.append(f'- **{_iss}**: {fp_by_issue[_iss]}')
    for _s in sorted(fp_by_section):
        md.append(f'- section **{_s}**: {fp_by_section[_s]}')
else:
    md.append('- none — zero FPs on the 16.')
for x in res_rows:
    md.append(f"| {x['id']} | {x['r0_full']} | {x['r1_full']} | {x['r0_clause_max']} | {x['r1_clause_max']} | {x['aux_residence_max']} | {x['r0_final']} | {x['residence_claim']} | {x['residence_gated']} | {x['residence_fired']} |")
md += ['', f"H2 check: residence fired={h2_res_fired}, PWDVA 17/18/19 kept={h2_pwdva_kept}.",
       f"H12 check: stays empty={h12_empty}.",
       '',
       'Escalates are scored BOTH as a boolean (v6 kind set) and as typed-kinds equality (strict + fact-excluded) — see Cell 8 header.',
       'H7/H15 escalate expectations RESOLVED (D3/D4, rater 2026-11-27); procedure/recovery kinds added.',
       'Prose forbidden mappings (Fp) NOT MEASURED in v6 (Cell 9 skipped); the 9.0 cap applies to any rating quoted from this run.',
       'Fresh-20 (sealed) is a MEASURED NUMBER ONLY, not a gate pass, until the G2 20/20 amendment is signed.',
       'Forbidden-after-recheck is 0 by construction when Stage 2 ran (Stage 2 drops those sections); '
       'quote forbidden-before-recheck / forbidden-proposed-before-gate as the live error counts.',
       'Prose-level forbidden mappings (Rajnesh / Satish / tenancy label) are NOT MEASURED here.',
       '']
with open('live-laya-colab-results-v6-brutal-regression.md', 'w') as f:
    f.write('\n'.join(md))
print('saved live-laya-colab-results-v6-brutal-regression.json and live-laya-colab-results-v6-brutal-regression.md')
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-brutal-regression.json')
_cf.download('live-laya-colab-results-v6-brutal-regression.md')
print('Downloaded both v6 regression result files. Send them back to Muse to record the live score.')


In [ ]:
# Batch B1 — brutal bank questions B-001..B-025 (25 questions). NO TUNING between batches.
# This cell ONLY runs the frozen pipeline (Cell 7 run_one_question) on the batch and scores
# it against the frozen bank GT. It must not modify any threshold, fact, gate, prompt or
# graph value, and no result from this batch may change any later batch. If this cell
# errors, file its partial output and resume from the NEXT batch — never re-run a filed batch.
_BATCH_QS = BANK_QUESTIONS[0:25]
_BATCH_RECS = []
for _q in _BATCH_QS:
    _t = {'id': _q['id'], 'question': _q['question'],
          'gt_ids': list(_q['gt_sections']), 'gt_labels': list(_q['gt_sections']),
          'facts': complete_facts(_q['question'])}
    _rec = run_one_question(_t)
    _BATCH_RECS.append(_rec)
    BRUTAL_RESULTS.append(_rec)
    print(f"{_q['id']}: kept={_rec['kept_ids']} court={_rec['court_live']} latency={_rec['latency_s']}s error={_rec['error']}")
_B1_ROWS, _B1_AGG = score_batch(bank_pairs(_BATCH_RECS), 'B1')
_B1_OUT = {'batch': 'B1', 'question_ids': [q['id'] for q in _BATCH_QS],
             'aggregate': _B1_AGG, 'per_question': _B1_ROWS,
             'raw_live_runs': _BATCH_RECS,
             'frozen_hashes': {'bank_sha256': '934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717', 'graph_file': GRAPH_FILE,
                             'notebook_version': 'v6-brutal'},
             'hardware': LOAD_INFO}
with open('live-laya-colab-results-v6-brutal-B1.json', 'w') as _f:
    json.dump(_B1_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-brutal-B1.json')
print(f"Batch B1 filed + downloaded: live-laya-colab-results-v6-brutal-B1.json — FILE IT before starting batch B2.")
_CUM_ROWS, _CUM_AGG = score_batch(bank_pairs(BRUTAL_RESULTS), 'cumulative-so-far')
print('CUMULATIVE so far: ' + str(_CUM_AGG))


In [ ]:
# Batch B2 — brutal bank questions B-026..B-050 (25 questions). NO TUNING between batches.
# This cell ONLY runs the frozen pipeline (Cell 7 run_one_question) on the batch and scores
# it against the frozen bank GT. It must not modify any threshold, fact, gate, prompt or
# graph value, and no result from this batch may change any later batch. If this cell
# errors, file its partial output and resume from the NEXT batch — never re-run a filed batch.
_BATCH_QS = BANK_QUESTIONS[25:50]
_BATCH_RECS = []
for _q in _BATCH_QS:
    _t = {'id': _q['id'], 'question': _q['question'],
          'gt_ids': list(_q['gt_sections']), 'gt_labels': list(_q['gt_sections']),
          'facts': complete_facts(_q['question'])}
    _rec = run_one_question(_t)
    _BATCH_RECS.append(_rec)
    BRUTAL_RESULTS.append(_rec)
    print(f"{_q['id']}: kept={_rec['kept_ids']} court={_rec['court_live']} latency={_rec['latency_s']}s error={_rec['error']}")
_B2_ROWS, _B2_AGG = score_batch(bank_pairs(_BATCH_RECS), 'B2')
_B2_OUT = {'batch': 'B2', 'question_ids': [q['id'] for q in _BATCH_QS],
             'aggregate': _B2_AGG, 'per_question': _B2_ROWS,
             'raw_live_runs': _BATCH_RECS,
             'frozen_hashes': {'bank_sha256': '934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717', 'graph_file': GRAPH_FILE,
                             'notebook_version': 'v6-brutal'},
             'hardware': LOAD_INFO}
with open('live-laya-colab-results-v6-brutal-B2.json', 'w') as _f:
    json.dump(_B2_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-brutal-B2.json')
print(f"Batch B2 filed + downloaded: live-laya-colab-results-v6-brutal-B2.json — FILE IT before starting batch B3.")
_CUM_ROWS, _CUM_AGG = score_batch(bank_pairs(BRUTAL_RESULTS), 'cumulative-so-far')
print('CUMULATIVE so far: ' + str(_CUM_AGG))


In [ ]:
# Batch B3 — brutal bank questions B-051..B-075 (25 questions). NO TUNING between batches.
# This cell ONLY runs the frozen pipeline (Cell 7 run_one_question) on the batch and scores
# it against the frozen bank GT. It must not modify any threshold, fact, gate, prompt or
# graph value, and no result from this batch may change any later batch. If this cell
# errors, file its partial output and resume from the NEXT batch — never re-run a filed batch.
_BATCH_QS = BANK_QUESTIONS[50:75]
_BATCH_RECS = []
for _q in _BATCH_QS:
    _t = {'id': _q['id'], 'question': _q['question'],
          'gt_ids': list(_q['gt_sections']), 'gt_labels': list(_q['gt_sections']),
          'facts': complete_facts(_q['question'])}
    _rec = run_one_question(_t)
    _BATCH_RECS.append(_rec)
    BRUTAL_RESULTS.append(_rec)
    print(f"{_q['id']}: kept={_rec['kept_ids']} court={_rec['court_live']} latency={_rec['latency_s']}s error={_rec['error']}")
_B3_ROWS, _B3_AGG = score_batch(bank_pairs(_BATCH_RECS), 'B3')
_B3_OUT = {'batch': 'B3', 'question_ids': [q['id'] for q in _BATCH_QS],
             'aggregate': _B3_AGG, 'per_question': _B3_ROWS,
             'raw_live_runs': _BATCH_RECS,
             'frozen_hashes': {'bank_sha256': '934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717', 'graph_file': GRAPH_FILE,
                             'notebook_version': 'v6-brutal'},
             'hardware': LOAD_INFO}
with open('live-laya-colab-results-v6-brutal-B3.json', 'w') as _f:
    json.dump(_B3_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-brutal-B3.json')
print(f"Batch B3 filed + downloaded: live-laya-colab-results-v6-brutal-B3.json — FILE IT before starting batch B4.")
_CUM_ROWS, _CUM_AGG = score_batch(bank_pairs(BRUTAL_RESULTS), 'cumulative-so-far')
print('CUMULATIVE so far: ' + str(_CUM_AGG))


In [ ]:
# Batch B4 — brutal bank questions B-076..B-100 (25 questions). NO TUNING between batches.
# This cell ONLY runs the frozen pipeline (Cell 7 run_one_question) on the batch and scores
# it against the frozen bank GT. It must not modify any threshold, fact, gate, prompt or
# graph value, and no result from this batch may change any later batch. If this cell
# errors, file its partial output and resume from the NEXT batch — never re-run a filed batch.
_BATCH_QS = BANK_QUESTIONS[75:100]
_BATCH_RECS = []
for _q in _BATCH_QS:
    _t = {'id': _q['id'], 'question': _q['question'],
          'gt_ids': list(_q['gt_sections']), 'gt_labels': list(_q['gt_sections']),
          'facts': complete_facts(_q['question'])}
    _rec = run_one_question(_t)
    _BATCH_RECS.append(_rec)
    BRUTAL_RESULTS.append(_rec)
    print(f"{_q['id']}: kept={_rec['kept_ids']} court={_rec['court_live']} latency={_rec['latency_s']}s error={_rec['error']}")
_B4_ROWS, _B4_AGG = score_batch(bank_pairs(_BATCH_RECS), 'B4')
_B4_OUT = {'batch': 'B4', 'question_ids': [q['id'] for q in _BATCH_QS],
             'aggregate': _B4_AGG, 'per_question': _B4_ROWS,
             'raw_live_runs': _BATCH_RECS,
             'frozen_hashes': {'bank_sha256': '934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717', 'graph_file': GRAPH_FILE,
                             'notebook_version': 'v6-brutal'},
             'hardware': LOAD_INFO}
with open('live-laya-colab-results-v6-brutal-B4.json', 'w') as _f:
    json.dump(_B4_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-brutal-B4.json')
print(f"Batch B4 filed + downloaded: live-laya-colab-results-v6-brutal-B4.json — FILE IT before starting batch B5.")
_CUM_ROWS, _CUM_AGG = score_batch(bank_pairs(BRUTAL_RESULTS), 'cumulative-so-far')
print('CUMULATIVE so far: ' + str(_CUM_AGG))


In [ ]:
# Batch B5 — brutal bank questions B-101..B-125 (25 questions). NO TUNING between batches.
# This cell ONLY runs the frozen pipeline (Cell 7 run_one_question) on the batch and scores
# it against the frozen bank GT. It must not modify any threshold, fact, gate, prompt or
# graph value, and no result from this batch may change any later batch. If this cell
# errors, file its partial output and resume from the NEXT batch — never re-run a filed batch.
_BATCH_QS = BANK_QUESTIONS[100:125]
_BATCH_RECS = []
for _q in _BATCH_QS:
    _t = {'id': _q['id'], 'question': _q['question'],
          'gt_ids': list(_q['gt_sections']), 'gt_labels': list(_q['gt_sections']),
          'facts': complete_facts(_q['question'])}
    _rec = run_one_question(_t)
    _BATCH_RECS.append(_rec)
    BRUTAL_RESULTS.append(_rec)
    print(f"{_q['id']}: kept={_rec['kept_ids']} court={_rec['court_live']} latency={_rec['latency_s']}s error={_rec['error']}")
_B5_ROWS, _B5_AGG = score_batch(bank_pairs(_BATCH_RECS), 'B5')
_B5_OUT = {'batch': 'B5', 'question_ids': [q['id'] for q in _BATCH_QS],
             'aggregate': _B5_AGG, 'per_question': _B5_ROWS,
             'raw_live_runs': _BATCH_RECS,
             'frozen_hashes': {'bank_sha256': '934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717', 'graph_file': GRAPH_FILE,
                             'notebook_version': 'v6-brutal'},
             'hardware': LOAD_INFO}
with open('live-laya-colab-results-v6-brutal-B5.json', 'w') as _f:
    json.dump(_B5_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-brutal-B5.json')
print(f"Batch B5 filed + downloaded: live-laya-colab-results-v6-brutal-B5.json — FILE IT before starting batch B6.")
_CUM_ROWS, _CUM_AGG = score_batch(bank_pairs(BRUTAL_RESULTS), 'cumulative-so-far')
print('CUMULATIVE so far: ' + str(_CUM_AGG))


In [ ]:
# Batch B6 — brutal bank questions B-126..B-150 (25 questions). NO TUNING between batches.
# This cell ONLY runs the frozen pipeline (Cell 7 run_one_question) on the batch and scores
# it against the frozen bank GT. It must not modify any threshold, fact, gate, prompt or
# graph value, and no result from this batch may change any later batch. If this cell
# errors, file its partial output and resume from the NEXT batch — never re-run a filed batch.
_BATCH_QS = BANK_QUESTIONS[125:150]
_BATCH_RECS = []
for _q in _BATCH_QS:
    _t = {'id': _q['id'], 'question': _q['question'],
          'gt_ids': list(_q['gt_sections']), 'gt_labels': list(_q['gt_sections']),
          'facts': complete_facts(_q['question'])}
    _rec = run_one_question(_t)
    _BATCH_RECS.append(_rec)
    BRUTAL_RESULTS.append(_rec)
    print(f"{_q['id']}: kept={_rec['kept_ids']} court={_rec['court_live']} latency={_rec['latency_s']}s error={_rec['error']}")
_B6_ROWS, _B6_AGG = score_batch(bank_pairs(_BATCH_RECS), 'B6')
_B6_OUT = {'batch': 'B6', 'question_ids': [q['id'] for q in _BATCH_QS],
             'aggregate': _B6_AGG, 'per_question': _B6_ROWS,
             'raw_live_runs': _BATCH_RECS,
             'frozen_hashes': {'bank_sha256': '934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717', 'graph_file': GRAPH_FILE,
                             'notebook_version': 'v6-brutal'},
             'hardware': LOAD_INFO}
with open('live-laya-colab-results-v6-brutal-B6.json', 'w') as _f:
    json.dump(_B6_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-brutal-B6.json')
print(f"Batch B6 filed + downloaded: live-laya-colab-results-v6-brutal-B6.json — FILE IT before starting batch B7.")
_CUM_ROWS, _CUM_AGG = score_batch(bank_pairs(BRUTAL_RESULTS), 'cumulative-so-far')
print('CUMULATIVE so far: ' + str(_CUM_AGG))


In [ ]:
# Batch B7 — brutal bank questions B-151..B-175 (25 questions). NO TUNING between batches.
# This cell ONLY runs the frozen pipeline (Cell 7 run_one_question) on the batch and scores
# it against the frozen bank GT. It must not modify any threshold, fact, gate, prompt or
# graph value, and no result from this batch may change any later batch. If this cell
# errors, file its partial output and resume from the NEXT batch — never re-run a filed batch.
_BATCH_QS = BANK_QUESTIONS[150:175]
_BATCH_RECS = []
for _q in _BATCH_QS:
    _t = {'id': _q['id'], 'question': _q['question'],
          'gt_ids': list(_q['gt_sections']), 'gt_labels': list(_q['gt_sections']),
          'facts': complete_facts(_q['question'])}
    _rec = run_one_question(_t)
    _BATCH_RECS.append(_rec)
    BRUTAL_RESULTS.append(_rec)
    print(f"{_q['id']}: kept={_rec['kept_ids']} court={_rec['court_live']} latency={_rec['latency_s']}s error={_rec['error']}")
_B7_ROWS, _B7_AGG = score_batch(bank_pairs(_BATCH_RECS), 'B7')
_B7_OUT = {'batch': 'B7', 'question_ids': [q['id'] for q in _BATCH_QS],
             'aggregate': _B7_AGG, 'per_question': _B7_ROWS,
             'raw_live_runs': _BATCH_RECS,
             'frozen_hashes': {'bank_sha256': '934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717', 'graph_file': GRAPH_FILE,
                             'notebook_version': 'v6-brutal'},
             'hardware': LOAD_INFO}
with open('live-laya-colab-results-v6-brutal-B7.json', 'w') as _f:
    json.dump(_B7_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-brutal-B7.json')
print(f"Batch B7 filed + downloaded: live-laya-colab-results-v6-brutal-B7.json — FILE IT before starting batch B8.")
_CUM_ROWS, _CUM_AGG = score_batch(bank_pairs(BRUTAL_RESULTS), 'cumulative-so-far')
print('CUMULATIVE so far: ' + str(_CUM_AGG))


In [ ]:
# Batch B8 — brutal bank questions B-176..B-200 (25 questions). NO TUNING between batches.
# This cell ONLY runs the frozen pipeline (Cell 7 run_one_question) on the batch and scores
# it against the frozen bank GT. It must not modify any threshold, fact, gate, prompt or
# graph value, and no result from this batch may change any later batch. If this cell
# errors, file its partial output and resume from the NEXT batch — never re-run a filed batch.
_BATCH_QS = BANK_QUESTIONS[175:200]
_BATCH_RECS = []
for _q in _BATCH_QS:
    _t = {'id': _q['id'], 'question': _q['question'],
          'gt_ids': list(_q['gt_sections']), 'gt_labels': list(_q['gt_sections']),
          'facts': complete_facts(_q['question'])}
    _rec = run_one_question(_t)
    _BATCH_RECS.append(_rec)
    BRUTAL_RESULTS.append(_rec)
    print(f"{_q['id']}: kept={_rec['kept_ids']} court={_rec['court_live']} latency={_rec['latency_s']}s error={_rec['error']}")
_B8_ROWS, _B8_AGG = score_batch(bank_pairs(_BATCH_RECS), 'B8')
_B8_OUT = {'batch': 'B8', 'question_ids': [q['id'] for q in _BATCH_QS],
             'aggregate': _B8_AGG, 'per_question': _B8_ROWS,
             'raw_live_runs': _BATCH_RECS,
             'frozen_hashes': {'bank_sha256': '934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717', 'graph_file': GRAPH_FILE,
                             'notebook_version': 'v6-brutal'},
             'hardware': LOAD_INFO}
with open('live-laya-colab-results-v6-brutal-B8.json', 'w') as _f:
    json.dump(_B8_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-brutal-B8.json')
print(f"Batch B8 filed + downloaded: live-laya-colab-results-v6-brutal-B8.json — FILE IT before starting batch B9.")
_CUM_ROWS, _CUM_AGG = score_batch(bank_pairs(BRUTAL_RESULTS), 'cumulative-so-far')
print('CUMULATIVE so far: ' + str(_CUM_AGG))


In [ ]:
# Batch B9 — brutal bank questions B-201..B-225 (25 questions). NO TUNING between batches.
# This cell ONLY runs the frozen pipeline (Cell 7 run_one_question) on the batch and scores
# it against the frozen bank GT. It must not modify any threshold, fact, gate, prompt or
# graph value, and no result from this batch may change any later batch. If this cell
# errors, file its partial output and resume from the NEXT batch — never re-run a filed batch.
_BATCH_QS = BANK_QUESTIONS[200:225]
_BATCH_RECS = []
for _q in _BATCH_QS:
    _t = {'id': _q['id'], 'question': _q['question'],
          'gt_ids': list(_q['gt_sections']), 'gt_labels': list(_q['gt_sections']),
          'facts': complete_facts(_q['question'])}
    _rec = run_one_question(_t)
    _BATCH_RECS.append(_rec)
    BRUTAL_RESULTS.append(_rec)
    print(f"{_q['id']}: kept={_rec['kept_ids']} court={_rec['court_live']} latency={_rec['latency_s']}s error={_rec['error']}")
_B9_ROWS, _B9_AGG = score_batch(bank_pairs(_BATCH_RECS), 'B9')
_B9_OUT = {'batch': 'B9', 'question_ids': [q['id'] for q in _BATCH_QS],
             'aggregate': _B9_AGG, 'per_question': _B9_ROWS,
             'raw_live_runs': _BATCH_RECS,
             'frozen_hashes': {'bank_sha256': '934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717', 'graph_file': GRAPH_FILE,
                             'notebook_version': 'v6-brutal'},
             'hardware': LOAD_INFO}
with open('live-laya-colab-results-v6-brutal-B9.json', 'w') as _f:
    json.dump(_B9_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-brutal-B9.json')
print(f"Batch B9 filed + downloaded: live-laya-colab-results-v6-brutal-B9.json — FILE IT before starting batch B10.")
_CUM_ROWS, _CUM_AGG = score_batch(bank_pairs(BRUTAL_RESULTS), 'cumulative-so-far')
print('CUMULATIVE so far: ' + str(_CUM_AGG))


In [ ]:
# Batch B10 — brutal bank questions B-226..B-230 (5 questions). NO TUNING between batches.
# This cell ONLY runs the frozen pipeline (Cell 7 run_one_question) on the batch and scores
# it against the frozen bank GT. It must not modify any threshold, fact, gate, prompt or
# graph value, and no result from this batch may change any later batch. If this cell
# errors, file its partial output and resume from the NEXT batch — never re-run a filed batch.
_BATCH_QS = BANK_QUESTIONS[225:230]
_BATCH_RECS = []
for _q in _BATCH_QS:
    _t = {'id': _q['id'], 'question': _q['question'],
          'gt_ids': list(_q['gt_sections']), 'gt_labels': list(_q['gt_sections']),
          'facts': complete_facts(_q['question'])}
    _rec = run_one_question(_t)
    _BATCH_RECS.append(_rec)
    BRUTAL_RESULTS.append(_rec)
    print(f"{_q['id']}: kept={_rec['kept_ids']} court={_rec['court_live']} latency={_rec['latency_s']}s error={_rec['error']}")
_B10_ROWS, _B10_AGG = score_batch(bank_pairs(_BATCH_RECS), 'B10')
_B10_OUT = {'batch': 'B10', 'question_ids': [q['id'] for q in _BATCH_QS],
             'aggregate': _B10_AGG, 'per_question': _B10_ROWS,
             'raw_live_runs': _BATCH_RECS,
             'frozen_hashes': {'bank_sha256': '934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717', 'graph_file': GRAPH_FILE,
                             'notebook_version': 'v6-brutal'},
             'hardware': LOAD_INFO}
with open('live-laya-colab-results-v6-brutal-B10.json', 'w') as _f:
    json.dump(_B10_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-brutal-B10.json')
print(f"Batch B10 filed + downloaded: live-laya-colab-results-v6-brutal-B10.json — FILE IT before starting batch B11.")
_CUM_ROWS, _CUM_AGG = score_batch(bank_pairs(BRUTAL_RESULTS), 'cumulative-so-far')
print('CUMULATIVE so far: ' + str(_CUM_AGG))


In [ ]:
# Sealed sets — upload + verify (read-only). NEVER edit these files.
# Upload sealed-fresh-20.json AND sealed-kannada-8.json in this one picker.
# Both are verified BEFORE any sealed question is scored:
#   fresh-20  SHA-256 1d8b5dc376e711c3ac6ead4424bb2d8ce52b07552b9a533ad86e0616dbcf8837 (20 questions, F1-F20)
#   kannada-8 SHA-256 2f652af1123a22441cc4158ddf480ef0981aa923858bbcb9dc3622f5e5c25853 (8 questions, KN1-KN8)
# A mismatch stops the run with an error.
# GATE NOTE (v6 section 7.2): the gate must be amended to 20/20 (G2) BEFORE any fresh run
# is SCORED as a gate pass. The amendment text is in the build note, NOT yet signed.
# Until the parent signs it, the fresh-20 numbers below are MEASURED NUMBERS ONLY,
# not a gate pass — the cells print that warning with the score.
print('Upload sealed-fresh-20.json and sealed-kannada-8.json now (file picker).')
_uploaded_sealed = files.upload()
print('uploaded:', list(_uploaded_sealed.keys()))
assert 'sealed-fresh-20.json' in _uploaded_sealed, 'sealed-fresh-20.json was not uploaded — upload it and re-run this cell.'
assert 'sealed-kannada-8.json' in _uploaded_sealed, 'sealed-kannada-8.json was not uploaded — upload it and re-run this cell.'
_fresh_sha = hashlib.sha256(_uploaded_sealed['sealed-fresh-20.json']).hexdigest()
_kn_sha = hashlib.sha256(_uploaded_sealed['sealed-kannada-8.json']).hexdigest()
print('fresh-20 SHA-256:', _fresh_sha)
print('kannada-8 SHA-256:', _kn_sha)
assert _fresh_sha == '1d8b5dc376e711c3ac6ead4424bb2d8ce52b07552b9a533ad86e0616dbcf8837', 'Fresh-20 SHA mismatch — wrong or changed file. STOP.'
assert _kn_sha == '2f652af1123a22441cc4158ddf480ef0981aa923858bbcb9dc3622f5e5c25853', 'Kannada-8 SHA mismatch — wrong or changed file. STOP.'
SEALED_FRESH = json.loads(_uploaded_sealed['sealed-fresh-20.json'].decode('utf-8'))
SEALED_KANNADA = json.loads(_uploaded_sealed['sealed-kannada-8.json'].decode('utf-8'))
assert len(SEALED_FRESH['questions']) == 20, 'Fresh-20 must hold 20 questions. STOP.'
assert len(SEALED_KANNADA['questions']) == 8, 'Kannada-8 must hold 8 questions. STOP.'
assert [q['id'] for q in SEALED_FRESH['questions']] == [f'F{i}' for i in range(1, 21)], 'Fresh-20 IDs are not F1..F20. STOP.'
assert [q['id'] for q in SEALED_KANNADA['questions']] == [f'KN{i}' for i in range(1, 9)], 'Kannada-8 IDs are not KN1..KN8. STOP.'
print('sealed sets OK: fresh-20 (20) + kannada-8 (8), SHAs verified. NEVER edit these files.')


In [ ]:
# Sealed fresh-20 — run + score. MEASURED NUMBER ONLY, NOT a gate pass (see header above).
# Scoring: exact / micro P/R/F1 / TP/FP/FN / court (lead-forum containment) /
# escalates boolean + kinds (mapped, see sealed_pairs in Cell 8).
# Court names normalized (Judicial Magistrate First Class -> Magistrate,
# Registrar of Births and Deaths -> Registrar). Section ids need NO renaming:
# the v5 live JSON shows the pipeline already emits hmga-1956-s6 /
# welfare-principle exactly as sealed GT does (identity mapping, verified).
# Sealed gt_labels are issue words, never used for scoring.
# COURT-CONVENTION NOTE: sealed maintenance-only questions expect Family Court
# (freeze note section 12) while the 16-set expects Magistrate for the same
# relief — the pipeline follows the 16-set convention, so sealed court scores
# will show systematic misses on maintenance-only questions (F1, F2, KN1).
# That is a GT-convention difference, not a pipeline change to make here.
# F20 RULE (freeze note section 11): F20 is out-of-scope with gt_escalates=True
# as a human-lawyer handoff; it is scored correct iff the system declines
# (empty kept, empty court), not as a failure.
# HMA 27 GT-RULE NOTE: F8, F9, F10, F16 carry hma-27 in GT frozen BEFORE D1-A
# (all proceeding_pending=False); the D2 gate drops it, so any hma-27 miss
# there is a GT-rule note, not a plain system error — listed in the aggregate
# as hma27_gt_note_ids.
_FRESH_RECS = []
for _q in SEALED_FRESH['questions']:
    _t = {'id': _q['id'], 'question': _q['question'],
          'gt_ids': list(_q['gt_sections']), 'gt_labels': list(_q['gt_sections']),
          'facts': complete_facts(_q['question'], partial=dict(_q.get('facts', {})))}
    _rec = run_one_question(_t)
    _FRESH_RECS.append(_rec)
    print(f"{_q['id']}: kept={_rec['kept_ids']} court={_rec['court_live']} latency={_rec['latency_s']}s error={_rec['error']}")
_FRESH_ROWS, _FRESH_AGG = score_batch(sealed_pairs(_FRESH_RECS, SEALED_FRESH), 'sealed-fresh-20')
_FRESH_OUT = {'set': 'sealed-fresh-20', 'aggregate': _FRESH_AGG, 'per_question': _FRESH_ROWS,
              'raw_live_runs': _FRESH_RECS,
              'frozen_hashes': {'fresh_sha256': '1d8b5dc376e711c3ac6ead4424bb2d8ce52b07552b9a533ad86e0616dbcf8837', 'graph_file': GRAPH_FILE,
                                'notebook_version': 'v6-brutal'},
              'hardware': LOAD_INFO,
              'gate_note': 'MEASURED NUMBER ONLY — not a gate pass until the G2 20/20 amendment is signed.'}
with open('live-laya-colab-results-v6-sealed-fresh-20.json', 'w') as _f:
    json.dump(_FRESH_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-sealed-fresh-20.json')
print('Sealed fresh-20 filed + downloaded: live-laya-colab-results-v6-sealed-fresh-20.json')
print('GATE NOTE: fresh-20 is a MEASURED NUMBER ONLY — not a gate pass until the G2 20/20 amendment is signed.')


In [ ]:
# Sealed Kannada-8 — run + score (same rules as fresh-20 above).
# KANNADA LIMITATION (Cell 7 note 12): the fact-derivation patterns are English-only,
# so Kannada questions fall back to default facts and the v5 gates veto almost
# everything. Kannada scores through this build measure the English-text gates,
# not the model. Model-extracted facts (v6 deep-dive section 2) are the proper fix.
# HMA 27 GT-RULE NOTE: KN3 carries hma-27 in GT frozen BEFORE D1-A
# (proceeding_pending=False); the D2 gate drops it — see hma27_gt_note_ids.
_KANNADA_RECS = []
for _q in SEALED_KANNADA['questions']:
    _t = {'id': _q['id'], 'question': _q['question'],
          'gt_ids': list(_q['gt_sections']), 'gt_labels': list(_q['gt_sections']),
          'facts': complete_facts(_q['question'], partial=dict(_q.get('facts', {})))}
    _rec = run_one_question(_t)
    _KANNADA_RECS.append(_rec)
    print(f"{_q['id']}: kept={_rec['kept_ids']} court={_rec['court_live']} latency={_rec['latency_s']}s error={_rec['error']}")
_KANNADA_ROWS, _KANNADA_AGG = score_batch(sealed_pairs(_KANNADA_RECS, SEALED_KANNADA), 'sealed-kannada-8')
_KANNADA_OUT = {'set': 'sealed-kannada-8', 'aggregate': _KANNADA_AGG, 'per_question': _KANNADA_ROWS,
                'raw_live_runs': _KANNADA_RECS,
                'frozen_hashes': {'kannada_sha256': '2f652af1123a22441cc4158ddf480ef0981aa923858bbcb9dc3622f5e5c25853', 'graph_file': GRAPH_FILE,
                                  'notebook_version': 'v6-brutal'},
                'hardware': LOAD_INFO}
with open('live-laya-colab-results-v6-sealed-kannada-8.json', 'w') as _f:
    json.dump(_KANNADA_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-sealed-kannada-8.json')
print('Sealed Kannada-8 filed + downloaded: live-laya-colab-results-v6-sealed-kannada-8.json')


In [ ]:
# Final — cumulative brutal scoring (230) + 16 regression + sealed summaries + pass/fail.
# This cell RE-SCORES filed records only (score_batch over BRUTAL_RESULTS) — it runs
# NO questions, changes NO threshold/fact/gate, and never re-runs a batch.
# Pass bar (brutal-test-200-plan section 3.2): exact >= 200/230, micro-F1 >= 0.95,
# micro P/R >= 0.95, mean P/R >= 0.90, court >= 200/230, escalates-kinds >= 200/230,
# silent 0, forbidden-before-recheck 0, prose forbidden 0 (NOT MEASURED in v6),
# model errors 0, no category below 75% exact. If any row fails, the brutal test
# FAILS as a whole — no partial pass.
# The brutal-bank rating uses the frozen gate formula with N=230 (plan section 3.1)
# and is ALWAYS quoted with "(brutal bank, n = 230)", never as the 10/10 gate score.
# Fp is NOT MEASURED (Cell 9 skipped) so the 9.0 cap applies to the brutal rating too.
assert len(BRUTAL_RESULTS) == 230, f'Final cell needs all 230 brutal records, got {len(BRUTAL_RESULTS)} — run batches B1-B10 first. STOP.'
_FINAL_ROWS, _FINAL_AGG = score_batch(bank_pairs(BRUTAL_RESULTS), 'brutal-230')
_Eb = _FINAL_AGG['exact_count'] / 230
_Rmb = _FINAL_AGG['mean_recall']
_Pmb = _FINAL_AGG['mean_precision']
_Cb = int(_FINAL_AGG['court_correct'].split('/')[0]) / 230
_Sb = int(_FINAL_AGG['escalates_bool_correct'].split('/')[0]) / 230
_Fbb = _FINAL_AGG['forbidden_before_recheck_total']
_brutal_raw = 10 * (0.60 * _Eb + 0.15 * _Rmb + 0.10 * _Pmb + 0.075 * _Cb + 0.075 * _Sb) - 0.5 * _Fbb
_brutal_raw = max(0.0, min(10.0, round(_brutal_raw, 1)))
_brutal_capped = min(_brutal_raw, 9.0)  # Fp NOT MEASURED -> cap 9.0
_lo, _hi = _wilson(_FINAL_AGG['exact_count'], 230)
_kinds_n = int(_FINAL_AGG['escalates_kinds_correct_strict'].split('/')[0])
_pass = {
    'exact >= 200/230': _FINAL_AGG['exact_count'] >= 200,
    'micro-F1 >= 0.95': _FINAL_AGG['micro_f1'] >= 0.95,
    'micro-P >= 0.95': _FINAL_AGG['micro_precision'] >= 0.95,
    'micro-R >= 0.95': _FINAL_AGG['micro_recall'] >= 0.95,
    'mean-P >= 0.90': _FINAL_AGG['mean_precision'] >= 0.90,
    'mean-R >= 0.90': _FINAL_AGG['mean_recall'] >= 0.90,
    'court >= 200/230': int(_FINAL_AGG['court_correct'].split('/')[0]) >= 200,
    'escalates-kinds >= 200/230': _kinds_n >= 200,
    'silent == 0': _FINAL_AGG['silent_count'] == 0,
    'forbidden-before-recheck == 0': _Fbb == 0,
    'prose forbidden == 0': 'NOT MEASURED in v6 (Cell 9 skipped)',
    'model errors == 0': _FINAL_AGG['model_errors'] == 0,
    'no category < 75% exact': _FINAL_AGG['per_category_min_exact'] >= 0.75,
}
_brutal_pass = all(v is True for v in _pass.values())
print('=== FINAL — brutal bank 230 (v6) ===')
print(f"exact {_FINAL_AGG['exact']} (95% Wilson interval {_lo}-{_hi}) | micro P/R/F1 {_FINAL_AGG['micro_precision']}/{_FINAL_AGG['micro_recall']}/{_FINAL_AGG['micro_f1']} TP/FP/FN {_FINAL_AGG['micro_tp_fp_fn']}")
print(f"court {_FINAL_AGG['court_correct']} | esc-bool {_FINAL_AGG['escalates_bool_correct']} | esc-kinds strict {_FINAL_AGG['escalates_kinds_correct_strict']} (fact-excl {_FINAL_AGG['escalates_kinds_correct_fact_excluded']})")
print(f"silent {_FINAL_AGG['silent_count']} { _FINAL_AGG['silent_ids']} | Fb {_Fbb} | errors {_FINAL_AGG['model_errors']} | per-cat min {_FINAL_AGG['per_category_min_exact']}")
print(f"per-category exact: {_FINAL_AGG['per_category_exact']}")
print(f"brutal-bank rating: raw {_brutal_raw} -> capped {_brutal_capped} (brutal bank, n = 230; Fp NOT MEASURED, cap 9.0 applies)")
print('--- pass/fail vs plan section 3.2 ---')
for _k, _v in _pass.items():
    print(f'  {_k}: {_v}')
print(f"BRUTAL VERDICT: {'PASS' if _brutal_pass else 'FAIL'} (no partial pass; prose row is NOT MEASURED, so a full pass is impossible in v6 — see build note)")
print()
print('=== 16-question REGRESSION (corrected v2 GT, 53 sections) ===')
print(f"exact {agg['exact_section_set_match']} | micro P/R {agg['micro_precision']}/{agg['micro_recall']} TP/FP/FN {agg['micro_tp_fp_fn']} | court {agg['court_correct']} | esc-bool {agg['escalates_correct']} | esc-kinds {agg['escalates_kinds_correct']} (fact-excl {agg['escalates_kinds_correct_fact_excluded']})")
print(f"gate-formula rating (16): raw {agg['gate_formula_rating_raw']} -> capped {agg['gate_formula_rating_capped_9.0']} (prose NOT MEASURED)")
print('Regression must read 16/16 — if it does not, the D1-A/D2 re-base has a live defect; file the run and diagnose offline, do not tune in Colab.')
print()
_fresh_agg = globals().get('_FRESH_AGG')
_kn_agg = globals().get('_KANNADA_AGG')
print('=== Sealed sets (measured numbers only) ===')
if _fresh_agg is not None:
    print(f"fresh-20: exact {_fresh_agg['exact']} | micro P/R/F1 {_fresh_agg['micro_precision']}/{_fresh_agg['micro_recall']}/{_fresh_agg['micro_f1']} | court {_fresh_agg['court_correct']} | esc-bool {_fresh_agg['escalates_bool_correct']} | esc-kinds {_fresh_agg['escalates_kinds_correct_strict']} | silent {_fresh_agg['silent_count']} | Fb {_fresh_agg['forbidden_before_recheck_total']} | errors {_fresh_agg['model_errors']}")
    print('  fresh-20 is a MEASURED NUMBER ONLY — not a gate pass until the G2 20/20 amendment is signed.')
else:
    print('fresh-20: NOT RUN (sealed cells were skipped).')
if _kn_agg is not None:
    print(f"kannada-8: exact {_kn_agg['exact']} | micro P/R/F1 {_kn_agg['micro_precision']}/{_kn_agg['micro_recall']}/{_kn_agg['micro_f1']} | court {_kn_agg['court_correct']} | esc-bool {_kn_agg['escalates_bool_correct']} | esc-kinds {_kn_agg['escalates_kinds_correct_strict']} | silent {_kn_agg['silent_count']} | Fb {_kn_agg['forbidden_before_recheck_total']} | errors {_kn_agg['model_errors']}")
else:
    print('kannada-8: NOT RUN (sealed cells were skipped).')
print()
_lat_all = [r['latency_s'] for r in BRUTAL_RESULTS if r['latency_s'] is not None]
print(f"G14 latency (brutal 230): mean {round(sum(_lat_all) / len(_lat_all), 2) if _lat_all else None}s (<=15s? {bool(_lat_all and sum(_lat_all) / len(_lat_all) <= 15)}) | max {round(max(_lat_all), 2) if _lat_all else None}s — expect FAIL on CPU (v5 mean was 61.9s); T4 unverified, no T4 number claimed.")
print('Gate rows (summary): G1 regression 16/16 required | G2 fresh-20 MEASURED ONLY (amendment unsigned) | G3 kannada-8 measured | G12 prose Fp NOT MEASURED | G14 latency, see above | G15 confirmation re-run NOT DONE.')
_FINAL_OUT = {'notebook_version': 'v6-brutal', 'brutal_230': _FINAL_AGG,
              'brutal_rating_raw': _brutal_raw, 'brutal_rating_capped_9.0': _brutal_capped,
              'brutal_wilson_95': [_lo, _hi], 'brutal_pass_bar': _pass, 'brutal_verdict': 'PASS' if _brutal_pass else 'FAIL',
              'regression_16': agg, 'regression_rows': rows,
              'sealed_fresh_20': _fresh_agg, 'sealed_kannada_8': _kn_agg,
              'per_question_brutal': _FINAL_ROWS, 'raw_live_runs_brutal': BRUTAL_RESULTS,
              'frozen_hashes': {'bank_sha256': '934a2b2265ac137c96939e1fbeca3682a037010e42f972bfd459c2fcac4ec717', 'graph_file': GRAPH_FILE},
              'hardware': LOAD_INFO}
with open('live-laya-colab-results-v6-brutal-final.json', 'w') as _f:
    json.dump(_FINAL_OUT, _f, indent=1, ensure_ascii=False)
from google.colab import files as _cf
_cf.download('live-laya-colab-results-v6-brutal-final.json')
print('Final filed + downloaded: live-laya-colab-results-v6-brutal-final.json')
